# 这一版改了什么

输出目录是 `pose_pca/20261106_v2`。上一版 `20261106` 不覆盖。

θ 的符号是这里定的，还没有和论文核对：从“自己的胸鳍指向对方胸鳍”的水平方向，转到“自己的胸鳍指向自己头部”的水平方向，逆时针为正（从上往下看）。0 表示头部指向对方，范围是 [-π, π]。-π 和 π 是相邻的。

`D_MAX = 200` 的单位是毫米，也就是 20 cm。坐标本来就是毫米。如果打开 `NORM_BL`，这个数要改成体长。

## 改过的单元

- 单元 1：假设 4、5、7 改成胸鳍中点坐标系，并写上 θ 的符号
- 单元 2：`RUN_NAME` 加上 `_v2`，新配置都在这个单元
- 单元 4：共转坐标、14 个特征的名字、PCA 符号、物理量、缺口和骨架阈值
- 单元 5：Step 0 的说明
- 单元 6：缺口表、缺口长度直方图、骨架距离直方图、对齐检查
- 单元 7：Step 1 的说明
- 单元 8：累积方差条形图、载荷热图、超过 `LOAD_THR` 的载荷表、对齐检查
- 单元 10：14 维还原成骨架；图题里的符号句。画法和指标没改
- 单元 13：Step 4 的说明
- 单元 14：`STATE_VARS`
- 单元 16：目录标签加上 `vars-pcs` 或 `vars-physical`；物理量模式下每个变量用自己的分格数。转移计数的算法没改
- 单元 22：K 和 P 必须是同一套特征顺序；余弦仍取绝对值；`params.json` 写入 Infomap 后端和版本

## 新增的单元

- 单元 0：本说明
- 单元 17：Infomap 的说明
- 单元 18：Infomap

## 新增的配置

- `PCA_STANDARDIZE = False`
- `FIX_PC_SIGN = True`
- `LOAD_THR = 0.3`
- `PC_GROUPS = {}`（看完载荷热图后自己填，程序不自动分组）
- `SKELETON_THRESH = None`（不画线、不剔除；看完直方图后再填）
- `STATE_VARS = "pcs"`（或 `"physical"`，只用于版本 A）
- `D_MAX = 200`（毫米，等于 20 cm）
- `N_BINS_D = 10`
- `N_BINS_THETA = 10`
- `SYMMETRIZE_M1M2 = False`
- `KEEP_SELF_LOOPS = True`
- `N_TRIALS = 500`
- `N_TRIALS_SCAN = 50`
- `INFOMAP_SEED = 20261106`
- `CLUSTER_SELECT = "top_coverage"`
- `COVERAGE = 0.95`
- `TAU_SCAN = [1, 2, 3, 5, 7, 10, 12, 15, 17, 20, 25, 30, 35]`
- `N_SHUFFLE = 100`

## 没改的部分

Step 3 的自相关、Step 5 的转移计数、Step 6 里除特征顺序和绝对余弦以外的比较，都保持原样。版本 B 仍是 27 个原始坐标，不做平移和旋转。Infomap 不读手动标记，也不对 `per_pc` 聚类。

## 后来加的单元

- 单元 19：Step 5b 补充图的说明
- 单元 20：簇×簇转移图、高分辨率状态图、稀有状态过滤、逐簇放大、可选的 html
- 单元 23：手动标记验证的说明
- 单元 24：验证代码。`DO_VALIDATION` 为 False 时不读标记

## 后来加的配置

- `DPI_BIG = 300`
- `FILTER_RARE = True`
- `MIN_VISITS = 50`
- `TOP_ZOOM = 6`
- `MAKE_HTML = False`
- `DO_VALIDATION = False`
- `BORIS_DIR`
- `DEV_SETS = ["K"]`
- `TEST_SETS = ["P"]`
- `MIN_PURITY = 0.5`
- `OVERLAP_THR = 0.5`
- `TOL_FRAMES = 5`
- `BOOT = 1000`
- `SHIFT_N = 100`
- `N_EXAMPLE = 2`
- `LABEL_PAIR_MAP = {}`
- `COURTSHIP_LABELS = []`
- `ATTACK_LABELS = []`


# Pose PCA for three fish (K, P, and K+P)

Two versions run inside every analysis set:

- **pairs**: FM1, FM2, and M1M2, each aligned, then 14-D PCA
- **triple**: 27 raw coordinates, PCA with no alignment

No behaviour labels are used. Figure text is in English. Comments added for this revision are in Chinese. The notebook does not decide whether K and P agree; it writes the numbers, tables, and figures.

## Assumptions

1. Analysis set K is the two development videos `20251225_K_1_1_FCMC` and `20251225_K_1_2_FCMC`. Set P is `20260112_P_1_1_FCMC` and `20260113_P_1_2_FCMC`. Day 3 of each group is the held-out test video and is not opened. KP concatenates the four videos and fits its own PCA.
2. F, M1, and M2 are the identity-ledger names. Within a group those names are the same three animals across days. Across K and P they are different animals. Frames the identity pass did not accept are already NaN. BORIS behaviour files are not read.
3. Coordinates are millimetres in the reconstruction frame. z is vertical. The optional body-length switch divides by a length, and axis labels then say body lengths.
4. Version A puts the origin at the midpoint of the two pectoral points. Z' stays aligned with Z. X' is the horizontal unit vector from fish 1's pectoral to fish 2's pectoral, and Y' completes a right-handed frame. The 14 inputs, in this order, are fish 1 pectoral x' and z', fish 1 head, fish 1 tail, fish 2 head, and fish 2 tail. Fish 2's pectoral is the mirror of fish 1's and is not an input.
5. After that rotation both pectoral y' values are 0, and the two pectoral x' values sum to 0. The same is true of z'. The notebook prints the maximum absolute residuals and the T x 14 shape.
6. A frame whose horizontal pectoral-pectoral distance is below `MIN_XY_MM` has no yaw and is left out of version A. Version B does not use that cut.
7. PCA subtracts the column mean. It divides by the column standard deviation only when PCA_STANDARDIZE is true; the default is false. When FIX_PC_SIGN is true, each component is flipped so the loading with the largest absolute value is positive, and the scores flip with it. Loading comparisons still use the absolute cosine.
8. Autocorrelation is computed inside each continuous segment, demeaned in that segment, and averaged with weight equal to the segment length. The 1/e time is the linear interpolation of the first downward crossing. If the curve never crosses, the table entry is empty.
9. Bin edges come from the minimum and maximum of the current analysis set. The maximum falls in the last bin. The same edges are used for every tau.
10. A joint state is outside the source grid if any of its PCs is outside that PC's edges. Outside frames are excluded from the occupancy distribution and from the transition counts. The Jensen-Shannon value is `scipy.spatial.distance.jensenshannon` (the distance) on the occupancy distributions renormalised over the in-range states.
11. The day split uses the videos of a group, ordered by day, split into two halves. With two videos that is one day versus the other day. Those days are the same three fish, so they are not independent samples.
12. For each K component the matched P component is the one with the largest absolute cosine, chosen independently. Two K components may match the same P component.
13. Pearson correlation of a heading angle is reported on the angle in degrees. The angle is circular, so a branch cut can weaken that correlation.
14. When STATE_VARS is "physical", D_PP is the 3D pectoral–pectoral distance. Each theta is the counterclockwise angle from that fish's pectoral-to-pectoral direction to its pectoral-to-head direction, viewed from above. Zero means the head points at the other fish. This sign was chosen in this notebook and has not been checked against the paper. M1M2 keeps the order (theta of M1, theta of M2).


In [1]:
from pathlib import Path
import json
import os
import time

os.chdir(r"C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline")

# All analysis parameters. Optional switches default to off, except the two
# discretisation modes and the within-group day reference, which default to on.

RUN_NAME = "20261106_v2"
OUTPUT_ROOT = Path(r"Z:\medaka\runs\behavior_features\pose_pca")

K_EXPERIMENTS = [
    "20251225_K_1_1_FCMC",
    "20251225_K_1_2_FCMC",
]
P_EXPERIMENTS = [
    "20260112_P_1_1_FCMC",
    "20260113_P_1_2_FCMC",
]

ENABLE_CLEAN = False
K_LO = {"head_pec": 0.70, "pec_tail": 0.65}
K_HI = {"head_pec": 1.25, "pec_tail": 1.20}
FILL_LIMIT = 8
SG_WINDOW = 7
SG_ORDER = 2

NORM_BL = False
MIN_XY_MM = 1.0

N_SHOW = 3
K_POSES = (-2, 0, 2)
N_ACF = 10
MAX_LAG_S = 5.0

PCS_FOR_STATE = [1, 2, 3]  # 1-based component numbers
N_BINS = 10
BIN_BY_PERCENTILE = False
PERCENTILE_LO = 0.5
PERCENTILE_HI = 99.5
DO_JOINT = True
DO_PER_PC = True
TAUS = [7, 17]

N_CMP = 6
DO_DAY_REFERENCE = True

# PCA 前不标准化。论文没有写这一步。
PCA_STANDARDIZE = False
# 每个 PC 里，绝对值最大的载荷取正号，得分一起翻转。
FIX_PC_SIGN = True
# 绝对值超过这个数的载荷写进 csv。分组看完热图后自己填，不自动分组。
LOAD_THR = 0.3
PC_GROUPS = {}

# 骨架阈值。None 表示不画线、不剔除。看完直方图后再填。
# 单位跟坐标一样：默认毫米；打开 NORM_BL 之后是体长。
# 例子: {"head_pec": (8.0, 20.0), "pec_tail": (10.0, 22.0)}，某一侧可以是 None。
SKELETON_THRESH = None

# "pcs" 沿用 PCS_FOR_STATE。"physical" 只用于版本 A。
STATE_VARS = "pcs"
# 物理量的距离上限。坐标是毫米，200 = 20 cm。打开 NORM_BL 后改成体长。
D_MAX = 200.0
N_BINS_D = 10
N_BINS_THETA = 10
# 只作用于 M1M2。打开后每一帧再加入对调的两个角度。
SYMMETRIZE_M1M2 = False

# Infomap。同一次分析，包括 τ 扫描，只用同一个后端。
KEEP_SELF_LOOPS = True
N_TRIALS = 500
N_TRIALS_SCAN = 50
INFOMAP_SEED = 20261106
CLUSTER_SELECT = "top_coverage"
COVERAGE = 0.95
TAU_SCAN = [1, 2, 3, 5, 7, 10, 12, 15, 17, 20, 25, 30, 35]
N_SHUFFLE = 100

# Step 5b 的补充图。原有的块对角图保留。
DPI_BIG = 300
FILTER_RARE = True
MIN_VISITS = 50
TOP_ZOOM = 6
MAKE_HTML = False

# 手动标记验证。默认不读 BORIS。LABEL_PAIR_MAP 填好后再把 DO_VALIDATION 打开。
DO_VALIDATION = False
BORIS_DIR = Path("archive/boris")
DEV_SETS = ["K"]
TEST_SETS = ["P"]
MIN_PURITY = 0.5
OVERLAP_THR = 0.5
TOL_FRAMES = 5
BOOT = 1000
SHIFT_N = 100
N_EXAMPLE = 2
# 键是版本名。subjects 用 BORIS 的 Subject。behaviors 把行为名映射成评估标签，大小写不敏感。
# FM1 对 M1 的标签，FM2 对 M2，M1M2 对两条雄鱼之间的标签。空字典表示还没填。
LABEL_PAIR_MAP = {}
COURTSHIP_LABELS = []
ATTACK_LABELS = []

PARAMS = {k: v for k, v in globals().items() if k.isupper() and k not in {"PARAMS"}}
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in PARAMS.items()}, indent=2, default=str))


{
  "RUN_NAME": "20261106_v2",
  "OUTPUT_ROOT": "Z:\\medaka\\runs\\behavior_features\\pose_pca",
  "K_EXPERIMENTS": [
    "20251225_K_1_1_FCMC",
    "20251225_K_1_2_FCMC"
  ],
  "P_EXPERIMENTS": [
    "20260112_P_1_1_FCMC",
    "20260113_P_1_2_FCMC"
  ],
  "ENABLE_CLEAN": false,
  "K_LO": {
    "head_pec": 0.7,
    "pec_tail": 0.65
  },
  "K_HI": {
    "head_pec": 1.25,
    "pec_tail": 1.2
  },
  "FILL_LIMIT": 8,
  "SG_WINDOW": 7,
  "SG_ORDER": 2,
  "NORM_BL": false,
  "MIN_XY_MM": 1.0,
  "N_SHOW": 3,
  "K_POSES": [
    -2,
    0,
    2
  ],
  "N_ACF": 10,
  "MAX_LAG_S": 5.0,
  "PCS_FOR_STATE": [
    1,
    2,
    3
  ],
  "N_BINS": 10,
  "BIN_BY_PERCENTILE": false,
  "PERCENTILE_LO": 0.5,
  "PERCENTILE_HI": 99.5,
  "DO_JOINT": true,
  "DO_PER_PC": true,
  "TAUS": [
    7,
    17
  ],
  "N_CMP": 6,
  "DO_DAY_REFERENCE": true,
  "PCA_STANDARDIZE": false,
  "FIX_PC_SIGN": true,
  "LOAD_THR": 0.3,
  "PC_GROUPS": {},
  "SKELETON_THRESH": null,
  "STATE_VARS": "pcs",
  "D_MAX": 200.0,
  "N_

## Functions used by the steps

Not an analysis step. Loading, alignment, PCA, binning, and file paths live here so Steps 0–6 stay linear.


In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import jensenshannon

from scripts.analysis import microstate as M

FISH = ("F", "M1", "M2")
FISH_COLOR = {"F": "#c0392b", "M1": "#1a5276", "M2": "#1e8449"}
PAIRS = (
    ("pairs_FM1", "F-M1", ("F", "M1")),
    ("pairs_FM2", "F-M2", ("F", "M2")),
    ("pairs_M1M2", "M1-M2", ("M1", "M2")),
)
SETS = {
    "K": list(K_EXPERIMENTS),
    "P": list(P_EXPERIMENTS),
    "KP": list(K_EXPERIMENTS) + list(P_EXPERIMENTS),
}
ROOT = OUTPUT_ROOT / RUN_NAME
STORE = {}  # filled by the steps and read by step 6


def base_tag():
    parts = ["clean" if ENABLE_CLEAN else "raw"]
    if NORM_BL:
        parts.append("norm_bl")
    return "_".join(parts)


def unit_name():
    return "body length" if NORM_BL else "mm"


def loc(analysis, version, tag, step):
    path = ROOT / analysis / version / tag / step
    path.mkdir(parents=True, exist_ok=True)
    return path


def loc6(version, step):
    path = ROOT / "K_vs_P" / version / step
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_fig(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(" saved", path)


def body_length(nodes):
    head, pec, tail = nodes[:, :, 0], nodes[:, :, 1], nodes[:, :, 2]
    length = np.nanmedian(np.linalg.norm(head - pec, axis=-1), axis=0)
    length = length + np.nanmedian(np.linalg.norm(pec - tail, axis=-1), axis=0)
    return length


def prepare_nodes(trial):
    nodes = np.array(trial["named"], dtype=np.float64, copy=True)
    if ENABLE_CLEAN:
        nodes, _removed, _med = M.clean_skeleton(nodes, K_LO, K_HI)
        nodes = M.fill_gaps(nodes, int(FILL_LIMIT), trial["blocked"])
        nodes = M.smooth_sg(nodes, int(SG_WINDOW), int(SG_ORDER))
    lengths = body_length(nodes)
    if NORM_BL:
        if not np.isfinite(lengths).all() or np.any(lengths <= 0):
            raise RuntimeError(f"body length is not usable: {lengths}")
        nodes = nodes / lengths[None, :, None, None]
    return nodes, lengths


def segment_ids(valid):
    seg = -np.ones(len(valid), dtype=np.int32)
    change = np.diff(np.concatenate([[0], valid.astype(np.int8), [0]]))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    for i, (a, b) in enumerate(zip(starts, ends)):
        seg[a:b] = i
    return seg, starts, ends


def load_videos():
    allowed = set(K_EXPERIMENTS + P_EXPERIMENTS)

    def allow_listed(experiment, final):
        # The frozen split file is not on this machine. Only the videos named
        # in the parameter cell are read. Test-day videos are not in that list.
        if experiment not in allowed:
            raise PermissionError(experiment)

    M.require_access = allow_listed
    videos = []
    for experiment in K_EXPERIMENTS + P_EXPERIMENTS:
        trial = M.load_trial(experiment, final=False)
        nodes, lengths = prepare_nodes(trial)
        group, day = M.parse_experiment(experiment)
        videos.append({
            "experiment": experiment,
            "group": "K" if group.startswith("K") else "P",
            "day": day,
            "fps": float(trial["fps"]),
            "frames": np.asarray(trial["frames"], dtype=np.int64),
            "nodes": nodes,
            "blocked": np.asarray(trial["blocked"], dtype=bool),
            "body_length": lengths,
        })
        finite = np.isfinite(nodes).all(axis=(1, 2, 3))
        print(f"{experiment}  frames {len(nodes)}  shape {nodes.shape}  "
              f"all-finite {int(finite.sum())}  fps {trial['fps']:.3f}  "
              f"body length ({unit_name()}) {np.round(lengths, 3)}")
        print(f"  coordinate range {np.nanmin(nodes):.3f} .. {np.nanmax(nodes):.3f}")
    return videos


def videos_for(analysis):
    wanted = set(SETS[analysis])
    return [v for v in VIDEOS if v["experiment"] in wanted]


def pair_feature(nodes, pair_name):
    # 版本 A：原点是两个胸鳍点的中点。Z' 保持原 Z。
    # X' 是“鱼1胸鳍指向鱼2胸鳍”在 XY 上的单位向量。Y' 取 (-X'_y, X'_x)，使 X' × Y' = +Z'。
    points = M.pair_points(nodes, pair_name)
    feat, pose, valid = align_pair_midpoint(points, MIN_XY_MM)
    return feat, pose, valid


def align_pair_midpoint(points, min_xy):
    pec1 = points[:, 1, :]
    pec2 = points[:, 4, :]
    origin = 0.5 * (pec1 + pec2)
    rel = points - origin[:, None, :]
    dxy = pec2[:, :2] - pec1[:, :2]
    dist = np.linalg.norm(dxy, axis=1)
    finite = np.isfinite(points).all(axis=(1, 2))
    valid = finite & np.isfinite(dist) & (dist >= min_xy)
    dist_safe = np.where(dist > 0, dist, 1.0)
    xx = dxy[:, 0] / dist_safe
    xy = dxy[:, 1] / dist_safe
    yx = -xy
    yy = xx
    xp = rel[:, :, 0] * xx[:, None] + rel[:, :, 1] * xy[:, None]
    yp = rel[:, :, 0] * yx[:, None] + rel[:, :, 1] * yy[:, None]
    zp = rel[:, :, 2]
    pose = np.stack([xp, yp, zp], axis=-1)
    # 顺序：鱼1胸鳍 x',z'，鱼1头，鱼1尾，鱼2头，鱼2尾。鱼2胸鳍是镜像，不进入 PCA。
    feat = np.column_stack([
        pose[:, 1, 0], pose[:, 1, 2],
        pose[:, 0, 0], pose[:, 0, 1], pose[:, 0, 2],
        pose[:, 2, 0], pose[:, 2, 1], pose[:, 2, 2],
        pose[:, 3, 0], pose[:, 3, 1], pose[:, 3, 2],
        pose[:, 5, 0], pose[:, 5, 1], pose[:, 5, 2],
    ])
    feat = np.array(feat, dtype=np.float64, copy=True)
    feat[~valid] = np.nan
    pose = np.array(pose, dtype=np.float64, copy=True)
    pose[~valid] = np.nan
    return feat, pose, valid


def reconstruct_pair(feat):
    # 由 14 维还原 6 个点。鱼2胸鳍 = 鱼1胸鳍关于原点的镜像。
    feat = np.asarray(feat, dtype=np.float64)
    single = feat.ndim == 1
    if single:
        feat = feat.reshape(1, -1)
    pose = np.empty((len(feat), 6, 3), dtype=np.float64)
    pose[:, 1, 0] = feat[:, 0]
    pose[:, 1, 1] = 0.0
    pose[:, 1, 2] = feat[:, 1]
    pose[:, 4, 0] = -feat[:, 0]
    pose[:, 4, 1] = 0.0
    pose[:, 4, 2] = -feat[:, 1]
    pose[:, 0, :] = feat[:, 2:5]
    pose[:, 2, :] = feat[:, 5:8]
    pose[:, 3, :] = feat[:, 8:11]
    pose[:, 5, :] = feat[:, 11:14]
    bad = ~np.isfinite(feat).all(axis=1)
    pose[bad] = np.nan
    return pose[0] if single else pose


def alignment_checks(pose, valid):
    use = valid & np.isfinite(pose).all(axis=(1, 2))
    if int(use.sum()) == 0:
        return {"n": 0, "max_abs_pec_y": np.nan, "max_abs_pec_x_sum": np.nan}
    pec_y = pose[use][:, [1, 4], 1]
    x_sum = pose[use][:, 1, 0] + pose[use][:, 4, 0]
    return {
        "n": int(use.sum()),
        "max_abs_pec_y": float(np.max(np.abs(pec_y))),
        "max_abs_pec_x_sum": float(np.max(np.abs(x_sum))),
    }


def triple_feature(nodes):
    valid = np.isfinite(nodes).all(axis=(1, 2, 3))
    feat = nodes.reshape(len(nodes), 27).copy()
    feat[~valid] = np.nan
    return feat, valid


def fit_pca(x):
    # 默认只减均值。PCA_STANDARDIZE 为 True 时再除以列标准差。
    mu = x.mean(axis=0)
    centered = x - mu
    scale = np.ones(x.shape[1], dtype=np.float64)
    if PCA_STANDARDIZE:
        scale = x.std(axis=0, ddof=1)
        scale[~np.isfinite(scale) | (scale == 0)] = 1.0
        centered = centered / scale
    _u, s, vt = np.linalg.svd(centered, full_matrices=False)
    eig = (s ** 2) / max(len(x) - 1, 1)
    total = float(eig.sum())
    explained = eig / total if total > 0 else np.zeros_like(eig)
    components = vt.T.copy()
    if PCA_STANDARDIZE:
        # 除回标准差后，(x - mu) @ components 仍是标准化空间里的得分。
        components = components / scale[:, None]
    if FIX_PC_SIGN:
        for j in range(components.shape[1]):
            k = int(np.argmax(np.abs(components[:, j])))
            if components[k, j] < 0:
                components[:, j] *= -1.0
    return mu, components, explained


def score_frames(feat, mu, components):
    scores = np.full((len(feat), components.shape[1]), np.nan, dtype=np.float64)
    ok = np.isfinite(feat).all(axis=1)
    scores[ok] = (feat[ok] - mu) @ components
    return scores


def coord_names(version, fish_names=None):
    if version == "triple":
        return [f"{fish}_{point}_{axis}"
                for fish in FISH
                for point in ("head", "pec", "tail")
                for axis in ("x", "y", "z")]
    if fish_names is None:
        raise ValueError("pair feature names need fish_names")
    a, b = fish_names
    # 固定顺序：鱼1胸鳍 x',z'，鱼1头，鱼1尾，鱼2头，鱼2尾。
    names = [f"{a}_pec_x'", f"{a}_pec_z'"]
    for fish in (a, b):
        for part in ("head", "tail"):
            for axis in ("x'", "y'", "z'"):
                names.append(f"{fish}_{part}_{axis}")
    return names


def physical_quantities(nodes, pair_name, min_xy):
    # D_PP 是两个胸鳍点的 3D 距离。
    # theta 是从“自己的胸鳍指向对方胸鳍”的水平方向，转到“自己的胸鳍指向自己头部”的水平方向。
    # 逆时针为正。0 表示头部指向对方。
    points = M.pair_points(nodes, pair_name)
    pec1, pec2 = points[:, 1], points[:, 4]
    head1, head2 = points[:, 0], points[:, 3]
    dist = np.linalg.norm(pec2 - pec1, axis=1)
    finite = np.isfinite(pec1).all(1) & np.isfinite(pec2).all(1)
    dist = np.where(finite, dist, np.nan)

    def theta(head, pec, other):
        heading = head[:, :2] - pec[:, :2]
        ref = other[:, :2] - pec[:, :2]
        nh = np.linalg.norm(heading, axis=1)
        nr = np.linalg.norm(ref, axis=1)
        cross = ref[:, 0] * heading[:, 1] - ref[:, 1] * heading[:, 0]
        dot = np.sum(ref * heading, axis=1)
        ang = np.arctan2(cross, dot)
        good = (nh >= min_xy) & (nr >= min_xy) & np.isfinite(ang)
        out = np.full(len(head), np.nan, dtype=np.float64)
        out[good] = ang[good]
        return out

    return dist, theta(head1, pec1, pec2), theta(head2, pec2, pec1)


def skeleton_distances(nodes):
    head, pec, tail = nodes[:, :, 0], nodes[:, :, 1], nodes[:, :, 2]
    head_pec = np.linalg.norm(head - pec, axis=-1)
    pec_tail = np.linalg.norm(pec - tail, axis=-1)
    return head_pec, pec_tail


def apply_skeleton_thresh(nodes):
    # 默认 None：不改数据。给了阈值才把超范围的那条鱼标成 NaN。
    if SKELETON_THRESH is None:
        return nodes, 0.0
    out = np.array(nodes, dtype=np.float64, copy=True)
    head_pec, pec_tail = skeleton_distances(out)
    bad = np.zeros(len(out), dtype=bool)
    for fish in range(out.shape[1]):
        bad_fish = np.zeros(len(out), dtype=bool)
        for key, series in (("head_pec", head_pec[:, fish]), ("pec_tail", pec_tail[:, fish])):
            bounds = SKELETON_THRESH.get(key)
            if not bounds:
                continue
            lo, hi = bounds
            if lo is not None:
                bad_fish |= np.isfinite(series) & (series < lo)
            if hi is not None:
                bad_fish |= np.isfinite(series) & (series > hi)
        out[bad_fish, fish] = np.nan
        bad |= bad_fish
    frac = float(bad.mean()) if len(bad) else 0.0
    return out, frac


def required_missing(nodes, version, pair_name):
    # 该版本要用的坐标里只要有一个 NaN，这一帧就算缺失。
    if version == "triple":
        return ~np.isfinite(nodes).all(axis=(1, 2, 3))
    points = M.pair_points(nodes, pair_name)
    return ~np.isfinite(points).all(axis=(1, 2))


def boolean_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0 or not bool(mask.any()):
        return np.zeros(0, dtype=np.int64)
    padded = np.concatenate([[False], mask, [False]])
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    return (ends - starts).astype(np.int64)


def state_tag(version):
    # 目录名加上 vars-pcs 或 vars-physical，避免两种状态写到同一处。
    physical = STATE_VARS == "physical" and version != "triple"
    if physical:
        tag = f"binsD{N_BINS_D}_th{N_BINS_THETA}_vars-physical"
        if version == "pairs_M1M2" and SYMMETRIZE_M1M2:
            tag += "_sym"
    else:
        tag = "bins%d_pcs%s_vars-pcs" % (N_BINS, "-".join(str(p) for p in PCS_FOR_STATE))
        if BIN_BY_PERCENTILE:
            tag += "_pctl"
    if NORM_BL:
        tag += "_norm_bl"
    return tag


INFOMAP_INFO = {}

def bend_deg(mouth, pec, tail):
    return np.degrees(M.bend_angle(mouth, pec, tail))


def heading_deg(mouth, pec):
    return np.degrees(np.arctan2(mouth[:, 1] - pec[:, 1], mouth[:, 0] - pec[:, 0]))


def pearson(a, b):
    use = np.isfinite(a) & np.isfinite(b)
    if use.sum() < 3:
        return np.nan
    if np.nanstd(a[use]) == 0 or np.nanstd(b[use]) == 0:
        return np.nan
    return float(np.corrcoef(a[use], b[use])[0, 1])


def acf_curve(scores, seg, fps, max_lag_s):
    max_lag = int(np.floor(max_lag_s * fps))
    acc = np.zeros(max_lag + 1)
    weight = np.zeros(max_lag + 1)
    ids = np.unique(seg[seg >= 0])
    for sid in ids:
        x = scores[seg == sid]
        x = x[np.isfinite(x)]
        n = len(x)
        if n < 3:
            continue
        x = x - x.mean()
        nfft = 1 << int(np.ceil(np.log2(2 * n - 1)))
        spec = np.fft.rfft(x, nfft)
        corr = np.fft.irfft(spec * np.conjugate(spec), nfft)[:n]
        if corr[0] <= 0:
            continue
        m = min(max_lag, n - 1)
        acc[:m + 1] += (corr[:m + 1] / corr[0]) * n
        weight[:m + 1] += n
    with np.errstate(invalid="ignore", divide="ignore"):
        curve = np.where(weight > 0, acc / weight, np.nan)
    return np.arange(max_lag + 1) / fps, curve, weight


def time_to_e(lag_s, curve):
    target = np.exp(-1)
    below = np.flatnonzero(np.isfinite(curve) & (curve <= target) & (lag_s > 0))
    if len(below) == 0:
        return np.nan
    i = int(below[0])
    if i == 0 or not np.isfinite(curve[i - 1]):
        return float(lag_s[i])
    y0, y1 = curve[i - 1], curve[i]
    if y0 == y1:
        return float(lag_s[i])
    frac = (y0 - target) / (y0 - y1)
    return float(lag_s[i - 1] + frac * (lag_s[i] - lag_s[i - 1]))


def bin_edges(values):
    use = values[np.isfinite(values)]
    if BIN_BY_PERCENTILE:
        lo, hi = np.percentile(use, [PERCENTILE_LO, PERCENTILE_HI])
    else:
        lo, hi = float(use.min()), float(use.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.linspace(lo, hi, N_BINS + 1)


def digitize(values, edges):
    out = np.full(len(values), -1, dtype=np.int32)
    inside = np.isfinite(values) & (values >= edges[0]) & (values <= edges[-1])
    out[inside] = np.digitize(values[inside], edges[1:-1], right=False)
    return out


def joint_from_bins(bin_cols, widths=None):
    state = np.zeros(len(bin_cols[0]), dtype=np.int32)
    bad = np.zeros(len(state), dtype=bool)
    scale = 1
    for i, col in enumerate(bin_cols):
        bad |= col < 0
        state = state + np.where(col < 0, 0, col).astype(np.int32) * scale
        width = N_BINS if widths is None else int(widths[i])
        scale *= width
    state[bad] = -1
    return state

def transition_matrix(states, seg, tau, n_states):
    if tau >= len(states):
        return sparse.csr_matrix((n_states, n_states), dtype=np.float64), 0
    left, right = states[:-tau], states[tau:]
    same = (seg[:-tau] == seg[tau:]) & (seg[:-tau] >= 0)
    ok = same & (left >= 0) & (right >= 0)
    counts = sparse.coo_matrix(
        (np.ones(int(ok.sum()), dtype=np.float64), (left[ok], right[ok])),
        shape=(n_states, n_states),
    ).tocsr()
    return counts, int(ok.sum())


def row_normalize(counts):
    prob = counts.tocsr(copy=True).astype(np.float64)
    row_sum = np.asarray(prob.sum(axis=1)).ravel()
    prob = sparse.diags(np.where(row_sum > 0, 1.0 / np.where(row_sum > 0, row_sum, 1.0), 0.0)) @ prob
    return prob.tocsr(), row_sum


def stay_fraction(counts):
    total = float(counts.sum())
    if total <= 0:
        return np.nan
    return float(counts.diagonal().sum()) / total


def occupancy(states, n_states):
    used = states[states >= 0]
    counts = np.bincount(used, minlength=n_states).astype(np.float64) if len(states) else np.zeros(n_states)
    return counts


def js_distance(p, q):
    if p.sum() <= 0 or q.sum() <= 0:
        return np.nan
    return float(jensenshannon(p / p.sum(), q / q.sum()))


def cosine_abs(a, b, n):
    left = a[:, :n]
    right = b[:, :n]
    return np.abs(left.T @ right)


def best_matches(sim):
    rows = []
    for i in range(sim.shape[0]):
        j = int(np.argmax(sim[i]))
        rows.append({"pc_row": i + 1, "pc_col": j + 1, "abs_cosine": float(sim[i, j])})
    return pd.DataFrame(rows)


print("helpers ready", "sets", {k: v for k, v in SETS.items()})


helpers ready sets {'K': ['20251225_K_1_1_FCMC', '20251225_K_1_2_FCMC'], 'P': ['20260112_P_1_1_FCMC', '20260113_P_1_2_FCMC'], 'KP': ['20251225_K_1_1_FCMC', '20251225_K_1_2_FCMC', '20260112_P_1_1_FCMC', '20260113_P_1_2_FCMC']}


## Step 0 — Read data and continuous segments

A segment is a run of frames in which every coordinate required by that version is finite. Later steps stay inside one segment and do not cross a NaN gap.

The gap table and the gap-length histograms are written once under `step0_gaps`. A missing frame has a NaN in any coordinate that version needs. The histogram separates accepted frames with a NaN from frames blocked by review or identity. The skeleton-distance figure does not choose a threshold. `SKELETON_THRESH = None` draws no line and removes no frames.


In [3]:
VIDEOS = load_videos()
TAG0 = base_tag()
print("parameter tag", TAG0)

# 骨架距离直方图。阈值默认是 None：不画线，也不从后面的步骤里剔除帧。
hp_parts = []
pt_parts = []
for video in VIDEOS:
    head_pec, pec_tail = skeleton_distances(video["nodes"])
    hp_parts.append(head_pec[np.isfinite(head_pec)])
    pt_parts.append(pec_tail[np.isfinite(pec_tail)])
hp_all = np.concatenate(hp_parts) if hp_parts else np.zeros(0)
pt_all = np.concatenate(pt_parts) if pt_parts else np.zeros(0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
removed_text = "no threshold; no frames removed"
if SKELETON_THRESH is not None:
    bad_frames = 0
    total_frames = 0
    for video in VIDEOS:
        _nodes, frac = apply_skeleton_thresh(video["nodes"])
        bad_frames += frac * len(video["nodes"])
        total_frames += len(video["nodes"])
    removed_text = f"frames outside the threshold: {bad_frames / max(total_frames, 1):.4%}"
for ax, data, key, label in (
    (axes[0], hp_all, "head_pec", "head-pec"),
    (axes[1], pt_all, "pec_tail", "pec-tail"),
):
    if len(data):
        ax.hist(data, bins=80, color="0.35")
    ax.set_xlabel(f"{label} distance ({unit_name()})")
    ax.set_ylabel("frames x fish")
    bounds = None if SKELETON_THRESH is None else SKELETON_THRESH.get(key)
    if bounds:
        lo, hi = bounds
        if lo is not None:
            ax.axvline(lo, color="k", ls="--", lw=1)
        if hi is not None:
            ax.axvline(hi, color="k", ls="--", lw=1)
axes[0].set_title("All videos, all fish")
fig.suptitle("Skeleton distances. " + removed_text + "\nThresholds are not chosen automatically.")
fig.tight_layout()
gap_dir = ROOT / "step0_gaps"
save_fig(fig, gap_dir / "skeleton_distances.png")
print(removed_text)

if SKELETON_THRESH is None:
    print("SKELETON_THRESH is None")
else:
    total_bad = 0
    total_n = 0
    for video in VIDEOS:
        new_nodes, frac = apply_skeleton_thresh(video["nodes"])
        video["nodes"] = new_nodes
        total_bad += frac * len(new_nodes)
        total_n += len(new_nodes)
        print(f"  removed {video['experiment']}: {frac:.4%}")
    print(f"removed overall: {total_bad / max(total_n, 1):.4%}")

for analysis, members in SETS.items():
    chosen = videos_for(analysis)
    print(f"\n== {analysis} videos {[v['experiment'] for v in chosen]}")
    for version, pair_name, fish_names in list(PAIRS) + [("triple", None, FISH)]:
        rows = []
        n_valid = 0
        n_frames = 0
        for video in chosen:
            if version == "triple":
                _feat, valid = triple_feature(video["nodes"])
            else:
                _feat, _pose, valid = pair_feature(video["nodes"], pair_name)
            seg, starts, ends = segment_ids(valid)
            video.setdefault("pack", {})[version] = {"valid": valid, "seg": seg}
            n_valid += int(valid.sum())
            n_frames += len(valid)
            for sid, a, b in zip(range(len(starts)), starts, ends):
                rows.append({
                    "analysis": analysis, "version": version, "experiment": video["experiment"],
                    "segment": sid, "start_frame": int(video["frames"][a]),
                    "end_frame": int(video["frames"][b - 1]), "n_frames": int(b - a),
                })
        table = pd.DataFrame(rows)
        path = loc(analysis, version, TAG0, "step0")
        table.to_csv(path / "segments.csv", index=False)
        lengths = table["n_frames"] if len(table) else pd.Series(dtype=float)
        print(f"{analysis} {version}: frames {n_frames}  valid {n_valid}  "
              f"segments {len(table)}  length min/median/max "
              f"{lengths.min() if len(lengths) else 0}/"
              f"{lengths.median() if len(lengths) else 0}/"
              f"{lengths.max() if len(lengths) else 0}")

# 缺口表按“连续缺失”统计。直方图在缺失原因切换的地方把缺口切开。
gap_rows = []
gap_lengths = {version: {"track": [], "review": []} for version, _p, _f in list(PAIRS) + [("triple", None, FISH)]}
for video in VIDEOS:
    for version, pair_name, _fish in list(PAIRS) + [("triple", None, FISH)]:
        missing = required_missing(video["nodes"], version, pair_name)
        review = missing & video["blocked"]
        track = missing & ~video["blocked"]
        gaps = boolean_runs(missing)
        finite_runs = boolean_runs(~missing)
        gap_lengths[version]["track"].append(boolean_runs(track))
        gap_lengths[version]["review"].append(boolean_runs(review))
        gap_rows.append({
            "experiment": video["experiment"],
            "version": version,
            "n_frames": int(len(missing)),
            "missing_percent": float(100.0 * missing.mean()) if len(missing) else np.nan,
            "mean_gap_frames": float(gaps.mean()) if len(gaps) else np.nan,
            "gap_length_p99_frames": float(np.percentile(gaps, 99)) if len(gaps) else np.nan,
            "n_finite_segments": int(len(finite_runs)),
            "longest_finite_segment_seconds": float(finite_runs.max() / video["fps"]) if len(finite_runs) else 0.0,
        })
gap_table = pd.DataFrame(gap_rows)
gap_table.to_csv(gap_dir / "gap_table.csv", index=False)
print("\ngap table")
print(gap_table.to_string(index=False))

for version, _pair, _fish in list(PAIRS) + [("triple", None, FISH)]:
    track = np.concatenate(gap_lengths[version]["track"]) if gap_lengths[version]["track"] else np.zeros(0)
    review = np.concatenate(gap_lengths[version]["review"]) if gap_lengths[version]["review"] else np.zeros(0)
    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    pieces = [x for x in (track, review) if len(x)]
    if pieces:
        hi = max(float(np.max(x)) for x in pieces)
        bins = np.linspace(0.5, hi + 0.5, int(min(max(hi, 1), 60)) + 1)
        if len(track):
            ax.hist(track, bins=bins, log=True, histtype="step", linewidth=1.6,
                    label="NaN on an accepted frame")
        if len(review):
            ax.hist(review, bins=bins, log=True, histtype="step", linewidth=1.6,
                    label="blocked by review or identity")
        ax.legend(fontsize=7)
    ax.set_xlabel("gap length (frames)")
    ax.set_ylabel("count (log)")
    note = "Gaps are split where the cause changes. The table counts unbroken missing runs."
    if len(track) == 0:
        note += " No accepted-frame gaps."
    if len(review) == 0:
        note += " No blocked gaps."
    ax.set_title(f"{version}\n{note}")
    save_fig(fig, gap_dir / f"gap_length_{version}.png")

print("\nAlignment check: max |pec y'|, max |pec1 x' + pec2 x'|")
for video in VIDEOS:
    for version, pair_name, fish_names in PAIRS:
        feat, pose, valid = pair_feature(video["nodes"], pair_name)
        chk = alignment_checks(pose, valid)
        print(f"{video['experiment']} {version} {fish_names[0]}->{fish_names[1]}  "
              f"feat {feat.shape}  n {chk['n']}  "
              f"max|pec y'| {chk['max_abs_pec_y']:.3e}  "
              f"max|pec x' sum| {chk['max_abs_pec_x_sum']:.3e}")


20251225_K_1_1_FCMC  frames 348600  shape (348600, 3, 3, 3)  all-finite 330760  fps 166.000  body length (mm) [24.631 24.374 25.231]
  coordinate range -30.753 .. 212.435


20251225_K_1_2_FCMC  frames 348600  shape (348600, 3, 3, 3)  all-finite 323710  fps 166.000  body length (mm) [24.769 24.446 25.312]
  coordinate range -10.812 .. 196.426


20260112_P_1_1_FCMC  frames 348600  shape (348600, 3, 3, 3)  all-finite 321996  fps 166.000  body length (mm) [23.769 25.816 25.441]
  coordinate range -24.363 .. 204.100


20260113_P_1_2_FCMC  frames 348600  shape (348600, 3, 3, 3)  all-finite 328754  fps 166.000  body length (mm) [24.087 25.878 25.659]
  coordinate range -38.917 .. 203.871
parameter tag raw


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\step0_gaps\skeleton_distances.png
no threshold; no frames removed
SKELETON_THRESH is None

== K videos ['20251225_K_1_1_FCMC', '20251225_K_1_2_FCMC']


K pairs_FM1: frames 697200  valid 658920  segments 1520  length min/median/max 1/10.0/13128


... (19 outputs left out)

20260112_P_1_1_FCMC pairs_M1M2 M1->M2  feat (348600, 14)  n 326497  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00


20260113_P_1_2_FCMC pairs_FM1 F->M1  feat (348600, 14)  n 329199  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00


20260113_P_1_2_FCMC pairs_FM2 F->M2  feat (348600, 14)  n 331193  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00


20260113_P_1_2_FCMC pairs_M1M2 M1->M2  feat (348600, 14)  n 331793  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00


## Step 1 — PCA

Version A places the origin at the pectoral midpoint, aligns the pectoral-to-pectoral direction with +X' in the horizontal plane, and runs PCA on the 14 coordinates listed in the feature names. Version B is the 27 raw coordinates, with no translation and no rotation.

The cumulative-variance bars use one colour unless `PC_GROUPS` is filled in by hand. The loading heatmap uses a zero-centred colour scale. `loadings_above_threshold.csv` lists entries whose absolute value exceeds `LOAD_THR`.


In [4]:
from matplotlib.patches import Patch

TAG1 = base_tag()
VAR_ROWS = []
print(f"PCA_STANDARDIZE={PCA_STANDARDIZE}  FIX_PC_SIGN={FIX_PC_SIGN}  LOAD_THR={LOAD_THR}")

# PC_GROUPS 留空就是单色。这里不根据载荷自动分组。
PC_GROUP_OF = {}
for group_name, pcs in PC_GROUPS.items():
    for pc in pcs:
        pc = int(pc)
        if pc in PC_GROUP_OF and PC_GROUP_OF[pc] != group_name:
            print(f"PC{pc} is in {PC_GROUP_OF[pc]} and {group_name}; the bar uses {group_name}")
        PC_GROUP_OF[pc] = group_name
PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#B279A2"]
NAME_COLOR = {name: PALETTE[i % len(PALETTE)] for i, name in enumerate(PC_GROUPS)}


def bar_colors(n_components):
    colors = []
    for i in range(1, n_components + 1):
        group_name = PC_GROUP_OF.get(i)
        colors.append(NAME_COLOR[group_name] if group_name else "#4C78A8")
    return colors


for analysis in SETS:
    chosen = videos_for(analysis)
    for version, pair_name, fish_names in list(PAIRS) + [("triple", None, FISH)]:
        blocks = []
        packed = []
        for video in chosen:
            if version == "triple":
                feat, valid = triple_feature(video["nodes"])
            else:
                feat, pose, valid = pair_feature(video["nodes"], pair_name)
                chk = alignment_checks(pose, valid)
                rec = reconstruct_pair(feat)
                both = valid & np.isfinite(pose).all(axis=(1, 2))
                if int(both.sum()):
                    recon_err = float(np.nanmax(np.abs(pose[both] - rec[both])))
                else:
                    recon_err = np.nan
                print(f"  {video['experiment']} {version} feat {tuple(feat.shape)}  "
                      f"max|pec y'| {chk['max_abs_pec_y']:.3e}  "
                      f"max|pec x' sum| {chk['max_abs_pec_x_sum']:.3e}  "
                      f"reconstruct max abs diff {recon_err:.3e}")
            blocks.append(feat[valid])
            packed.append((video, feat, valid))
        x = np.concatenate(blocks, axis=0)
        mu, components, explained = fit_pca(x)
        names = coord_names(version, None if version == "triple" else fish_names)
        if x.shape[1] != len(names):
            raise RuntimeError(f"{version} matrix {x.shape} does not match {len(names)} names")
        print(f"\n{analysis} {version}: valid rows {len(x)}  PCA matrix {tuple(x.shape)}  "
              f"value range {x.min():.4f} .. {x.max():.4f}")
        print(f"  features: {names}")
        print(f"  variance of first {min(6, len(explained))} PCs (%) "
              f"{np.round(100 * explained[:6], 3)}  cumulative {100 * explained.cumsum()[-1]:.2f}%")
        load = pd.DataFrame(components, index=names,
                            columns=[f"PC{i + 1}" for i in range(components.shape[1])])
        out = loc(analysis, version, TAG1, "step1")
        load.to_csv(out / "loadings.csv")
        pd.DataFrame({
            "pc": np.arange(1, len(explained) + 1),
            "explained_ratio": explained,
            "cumulative": np.cumsum(explained),
        }).to_csv(out / "variance.csv", index=False)
        np.savez(out / "pca_model.npz", mean=mu, components=components, explained=explained)

        cum_pct = 100.0 * np.cumsum(explained)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(np.arange(1, len(cum_pct) + 1), cum_pct, color=bar_colors(len(cum_pct)), width=0.85)
        ymin = 50.0
        if float(np.min(cum_pct)) < ymin:
            ymin = max(0.0, float(np.min(cum_pct)) - 5.0)
        ax.set_ylim(ymin, 100)
        ax.set_xlabel("PC")
        ax.set_ylabel("cumulative explained variance (%)")
        title = f"{analysis} {version}"
        if version == "triple":
            title += "\nraw coordinates, no translation or rotation"
        else:
            title += "\n14 coordinates, pectoral midpoint, yaw removed"
        ax.set_title(title)
        if PC_GROUPS:
            handles = [Patch(facecolor=NAME_COLOR[name], label=name) for name in PC_GROUPS]
            ax.legend(handles=handles, fontsize=8, title="PC_GROUPS")
        save_fig(fig, out / "cumulative_variance.png")

        limit = float(np.max(np.abs(components)))
        if limit == 0:
            limit = 1.0
        fig, ax = plt.subplots(figsize=(max(8.0, 0.46 * components.shape[1] + 2.0),
                                        max(5.5, 0.34 * len(names) + 1.6)))
        im = ax.imshow(components, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
        fig.colorbar(im, ax=ax, fraction=0.046, label="loading")
        ax.set_yticks(np.arange(len(names)), names, fontsize=7)
        xticklabels = []
        for i in range(1, components.shape[1] + 1):
            group_name = PC_GROUP_OF.get(i, "")
            xticklabels.append(f"PC{i}" if not group_name else f"PC{i}\n({group_name})")
        ax.set_xticks(np.arange(components.shape[1]), xticklabels, fontsize=7)
        ax.set_xlabel("component")
        ax.set_title(f"{analysis} {version} loadings")
        save_fig(fig, out / "loadings_heatmap.png")

        high_rows = []
        for j in range(components.shape[1]):
            for i, name in enumerate(names):
                value = float(components[i, j])
                if abs(value) > LOAD_THR:
                    high_rows.append({
                        "pc": j + 1, "feature": name,
                        "loading": value, "abs_loading": abs(value),
                    })
        high = pd.DataFrame(high_rows, columns=["pc", "feature", "loading", "abs_loading"])
        if len(high):
            high = high.sort_values(["pc", "abs_loading"], ascending=[True, False])
        high.to_csv(out / "loadings_above_threshold.csv", index=False)
        print(f"  loadings above {LOAD_THR}: {len(high)} entries")

        per_video = []
        for video, feat, valid in packed:
            scores = score_frames(feat, mu, components)
            seg, _s, _e = segment_ids(valid)
            per_video.append({
                "experiment": video["experiment"], "frames": video["frames"],
                "scores": scores, "seg": seg, "fps": video["fps"], "feat": feat,
            })
            np.savez_compressed(
                out / f"scores_{video['experiment']}.npz",
                frames=video["frames"], scores=scores.astype(np.float32), seg=seg, fps=video["fps"],
            )
        key = (analysis, version)
        STORE[key] = {
            "mu": mu, "components": components, "explained": explained,
            "videos": per_video, "fish": fish_names, "pair": pair_name,
            "feature_names": names,
        }
        for i, ratio in enumerate(explained, start=1):
            VAR_ROWS.append({"analysis": analysis, "version": version, "pc": i, "explained_ratio": ratio})
        print(f"  saved scores for {len(per_video)} videos, components {components.shape}")

pd.DataFrame(VAR_ROWS).to_csv(ROOT / "variance_long.csv", index=False)


PCA_STANDARDIZE=False  FIX_PC_SIGN=True  LOAD_THR=0.3


  20251225_K_1_1_FCMC pairs_FM1 feat (348600, 14)  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00  reconstruct max abs diff 1.421e-14


  20251225_K_1_2_FCMC pairs_FM1 feat (348600, 14)  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00  reconstruct max abs diff 1.421e-14

K pairs_FM1: valid rows 658920  PCA matrix (658920, 14)  value range -128.1772 .. 126.5447
  features: ["F_pec_x'", "F_pec_z'", "F_head_x'", "F_head_y'", "F_head_z'", "F_tail_x'", "F_tail_y'", "F_tail_z'", "M1_head_x'", "M1_head_y'", "M1_head_z'", "M1_tail_x'", "M1_tail_y'", "M1_tail_z'"]
  variance of first 6 PCs (%) [61.919 33.374  1.266  1.041  0.976  0.616]  cumulative 100.00%


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\raw\step1\cumulative_variance.png
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\raw\step1\loadings_heatmap.png
  loadings above 0.3: 40 entries


  saved scores for 2 videos, components (14, 14)


  20251225_K_1_1_FCMC pairs_FM2 feat (348600, 14)  max|pec y'| 1.421e-14  max|pec x' sum| 0.000e+00  reconstruct max abs diff 1.421e-14


... (42 outputs left out)

  saved scores for 4 videos, components (14, 14)



KP triple: valid rows 1305220  PCA matrix (1305220, 27)  value range -38.9165 .. 212.4352
  features: ['F_head_x', 'F_head_y', 'F_head_z', 'F_pec_x', 'F_pec_y', 'F_pec_z', 'F_tail_x', 'F_tail_y', 'F_tail_z', 'M1_head_x', 'M1_head_y', 'M1_head_z', 'M1_pec_x', 'M1_pec_y', 'M1_pec_z', 'M1_tail_x', 'M1_tail_y', 'M1_tail_z', 'M2_head_x', 'M2_head_y', 'M2_head_z', 'M2_pec_x', 'M2_pec_y', 'M2_pec_z', 'M2_tail_x', 'M2_tail_y', 'M2_tail_z']
  variance of first 6 PCs (%) [22.404 16.205 14.777  9.667  9.239  7.751]  cumulative 100.00%
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\raw\step1\cumulative_variance.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\raw\step1\loadings_heatmap.png
  loadings above 0.3: 101 entries


  saved scores for 4 videos, components (27, 27)


## Step 2 — Draw the components

Each drawn pose is the mean pose plus k times that component's score standard deviation times its loading, for k = −2, 0, +2. Other components stay at the mean. The sign of a component is arbitrary.


In [5]:
TAG2 = f"pcs1-{N_SHOW}"
if NORM_BL:
    TAG2 += "_norm_bl"

def pose_at(mu, components, scores, index):
    sigma = float(np.nanstd(scores[:, index], ddof=1))
    loading = components[:, index]
    poses = []
    for k in K_POSES:
        poses.append(mu + k * sigma * loading)
    return poses, sigma


def metrics_from_points(points, fish_names):
    # points: (n_fish, 3 points, 3)
    out = {}
    pec = points[:, 1]
    for i, a in enumerate(fish_names):
        for j, b in enumerate(fish_names):
            if j <= i:
                continue
            out[f"pec_distance_{a}_{b}"] = float(np.linalg.norm(pec[j] - pec[i]))
        out[f"bend_deg_{a}"] = float(bend_deg(points[i, 0][None], points[i, 1][None], points[i, 2][None])[0])
        out[f"heading_deg_{a}"] = float(heading_deg(points[i, 0][None], points[i, 1][None])[0])
    return out


def draw_pc(points_by_k, fish_names, title, path, table):
    fig = plt.figure(figsize=(12, 8))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
    axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
    styles = {K_POSES[0]: (":", 0.45), 0: ("-", 1.0), K_POSES[-1]: ("--", 0.85)}
    for ax, (ix, iy, xlab, ylab) in zip(axes, ((0, 1, "x", "y"), (0, 2, "x", "z"))):
        for k, pts in points_by_k.items():
            ls, alpha = styles[k]
            for fish_i, fish in enumerate(fish_names):
                chain = pts[fish_i]
                ax.plot(chain[:, ix], chain[:, iy], ls=ls, alpha=alpha,
                        color=FISH_COLOR[fish],
                        label=f"{fish} k={k:+.0f}" if ax is axes[0] else None)
                ax.scatter(chain[:, ix], chain[:, iy], s=12, color=FISH_COLOR[fish], alpha=alpha)
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_xlabel(f"{xlab} ({unit_name()})")
        ax.set_ylabel(f"{ylab} ({unit_name()})")
        ax.set_title("XY" if iy == 1 else "XZ")
    axes[0].legend(fontsize=7, ncol=3, loc="best")
    ax_t = fig.add_subplot(grid[1, :])
    ax_t.axis("off")
    show = table.copy()
    show.columns = [str(c) for c in show.columns]
    for col in show.columns:
        if col != "quantity":
            show[col] = show[col].map(lambda v: "" if not np.isfinite(v) else f"{float(v):.3f}")
    ax_t.table(cellText=show.values, colLabels=list(show.columns), loc="center")
    sign_note = "Largest |loading| is set positive." if FIX_PC_SIGN else "PC sign is arbitrary."
    fig.suptitle(title + f"\n{sign_note} k = 0 is the mean pose.")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    save_fig(fig, path)


for analysis in SETS:
    for version, _pair_name, fish_names in list(PAIRS) + [("triple", None, FISH)]:
        item = STORE[(analysis, version)]
        mu, components = item["mu"], item["components"]
        scores = np.concatenate([v["scores"] for v in item["videos"]], axis=0)
        names = coord_names(version, None if version == "triple" else fish_names)
        corr_rows = []
        if version == "triple":
            pose_stack = np.concatenate([v["feat"].reshape(len(v["feat"]), 3, 3, 3) for v in item["videos"]])
            fish_pts = pose_stack
        else:
            pose_stack = np.concatenate([reconstruct_pair(v["feat"]) for v in item["videos"]])
            fish_pts = np.stack([pose_stack[:, 0:3], pose_stack[:, 3:6]], axis=1)
        quantity = {}
        pec = fish_pts[:, :, 1, :]
        for i, a in enumerate(fish_names):
            for j, b in enumerate(fish_names):
                if j <= i:
                    continue
                quantity[f"pec_distance_{a}_{b}"] = np.linalg.norm(pec[:, j] - pec[:, i], axis=-1)
            quantity[f"bend_deg_{a}"] = bend_deg(fish_pts[:, i, 0], fish_pts[:, i, 1], fish_pts[:, i, 2])
            quantity[f"heading_deg_{a}"] = heading_deg(fish_pts[:, i, 0], fish_pts[:, i, 1])
        if version == "triple":
            quantity["centroid_x"] = fish_pts[:, :, :, 0].mean(axis=(1, 2))
            quantity["centroid_y"] = fish_pts[:, :, :, 1].mean(axis=(1, 2))
            quantity["centroid_z"] = fish_pts[:, :, :, 2].mean(axis=(1, 2))
            for i, fish in enumerate(fish_names):
                for axis, letter in enumerate("xyz"):
                    quantity[f"pec_{fish}_{letter}"] = fish_pts[:, i, 1, axis]
        n_pc = min(scores.shape[1], max(N_SHOW, N_ACF, N_CMP))
        for qname, series in quantity.items():
            for j in range(n_pc):
                corr_rows.append({
                    "quantity": qname, "pc": j + 1,
                    "pearson_r": pearson(series, scores[:, j]),
                })
        out = loc(analysis, version, TAG2, "step2")
        corr = pd.DataFrame(corr_rows)
        corr.to_csv(out / "pc_quantity_correlation.csv", index=False)
        print(f"\n{analysis} {version} correlation rows {len(corr)}  "
              f"score shape {scores.shape}  finite scores {int(np.isfinite(scores[:, 0]).sum())}")
        if version == "triple":
            share_rows = []
            for j in range(components.shape[1]):
                mag = np.abs(components[:, j]).reshape(3, 3, 3)
                total = mag.sum()
                share_rows.append({
                    "pc": j + 1,
                    "abs_share_x": float(mag[:, :, 0].sum() / total),
                    "abs_share_y": float(mag[:, :, 1].sum() / total),
                    "abs_share_z": float(mag[:, :, 2].sum() / total),
                })
            pd.DataFrame(share_rows).to_csv(out / "loading_xyz_share.csv", index=False)
        for index in range(N_SHOW):
            poses, sigma = pose_at(mu, components, scores, index)
            points_by_k = {}
            metric_rows = []
            for vec, k in zip(poses, K_POSES):
                if version == "triple":
                    pts = vec.reshape(3, 3, 3)
                else:
                    flat = reconstruct_pair(vec)
                    pts = np.stack([flat[0:3], flat[3:6]], axis=0)
                points_by_k[k] = pts
                row = metrics_from_points(pts, fish_names)
                row["k"] = k
                metric_rows.append(row)
            metric = pd.DataFrame(metric_rows).set_index("k").T.reset_index().rename(columns={"index": "quantity"})
            metric.to_csv(out / f"pc{index + 1}_pose_metrics.csv", index=False)
            title = f"{analysis} {version} PC{index + 1}   sigma = {sigma:.4f} {unit_name()}"
            if version == "triple":
                title += "\nMean pose averages different orientations and can look shortened. Reference only."
                title += "\nNo translation or rotation was removed."
            draw_pc(points_by_k, fish_names, title, out / f"pc{index + 1}_pose.png", metric)
            print(f"  PC{index + 1} sigma {sigma:.4f}")



K pairs_FM1 correlation rows 50  score shape (697200, 14)  finite scores 658920


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\pcs1-3\step2\pc1_pose.png
  PC1 sigma 90.7905


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\pcs1-3\step2\pc2_pose.png
  PC2 sigma 66.6548


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\pcs1-3\step2\pc3_pose.png
  PC3 sigma 12.9833



K pairs_FM2 correlation rows 50  score shape (697200, 14)  finite scores 657010


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM2\pcs1-3\step2\pc1_pose.png
  PC1 sigma 64.9106


... (38 outputs left out)


KP triple correlation rows 210  score shape (1394400, 27)  finite scores 1305220


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\pcs1-3\step2\pc1_pose.png
  PC1 sigma 156.8574


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\pcs1-3\step2\pc2_pose.png
  PC2 sigma 133.4027


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\pcs1-3\step2\pc3_pose.png
  PC3 sigma 127.3902


## Step 3 — Autocorrelation

Each component score is scaled to unit variance. The autocorrelation is computed inside each continuous segment and averaged with weight equal to the segment length.


In [6]:
TAG3 = f"acf{int(MAX_LAG_S)}s"
if NORM_BL:
    TAG3 += "_norm_bl"

def pooled_acf(videos, index):
    acc = None
    weight = None
    lag_s = None
    for video in videos:
        lag, curve, w = acf_curve(video["scores"][:, index], video["seg"], video["fps"], MAX_LAG_S)
        if lag_s is None:
            lag_s, acc, weight = lag, np.zeros_like(curve), np.zeros_like(w)
        if len(curve) != len(acc):
            raise RuntimeError("fps differs, so the lag grid differs; the videos in one set must share a frame rate")
        use = np.isfinite(curve) & (w > 0)
        acc[use] += curve[use] * w[use]
        weight[use] += w[use]
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(weight > 0, acc / weight, np.nan)
    return lag_s, out

for analysis in SETS:
    for version, _pair, _fish in list(PAIRS) + [("triple", None, FISH)]:
        item = STORE[(analysis, version)]
        n_comp = item["videos"][0]["scores"].shape[1]
        n_plot = min(N_ACF, n_comp)
        lag_s = None
        curves = []
        tau_e = []
        for index in range(n_comp):
            # Standardise this component to unit variance inside the analysis set.
            column = np.concatenate([v["scores"][:, index] for v in item["videos"]])
            scale = np.nanstd(column, ddof=1)
            scaled_videos = []
            for video in item["videos"]:
                copied = dict(video)
                scores = video["scores"].copy()
                scores[:, index] = scores[:, index] / scale
                copied["scores"] = scores
                scaled_videos.append(copied)
            lag_s, curve = pooled_acf(scaled_videos, index)
            curves.append(curve)
            tau_e.append(time_to_e(lag_s, curve))
        curves = np.vstack(curves)
        tau_e = np.asarray(tau_e, dtype=float)
        item["acf_lag_s"] = lag_s
        item["acf"] = curves
        item["tau_e"] = tau_e
        out = loc(analysis, version, TAG3, "step3")
        pd.DataFrame({
            "pc": np.arange(1, n_comp + 1),
            "time_to_1_over_e_s": tau_e,
        }).to_csv(out / "correlation_time.csv", index=False)
        np.savez(out / "acf.npz", lag_s=lag_s, acf=curves, time_to_1_over_e_s=tau_e)
        fig, axes = plt.subplots(2, 5, figsize=(14, 5), sharex=True, sharey=True)
        for ax, index in zip(axes.ravel(), range(n_plot)):
            ax.plot(lag_s, curves[index], color="0.15", lw=1)
            ax.axhline(np.exp(-1), color="0.5", lw=0.6, ls="--")
            if np.isfinite(tau_e[index]):
                ax.axvline(tau_e[index], color="#c0392b", lw=0.7)
            ax.set_title(f"PC{index + 1}  {tau_e[index]:.3f} s" if np.isfinite(tau_e[index]) else f"PC{index + 1}")
        for ax in axes[-1]:
            ax.set_xlabel("lag (s)")
        axes[0, 0].set_ylabel("ACF")
        fig.suptitle(f"{analysis} {version}. Red line: first drop to 1/e. PC sign is arbitrary.")
        fig.tight_layout()
        save_fig(fig, out / "acf.png")
        print(f"{analysis} {version} 1/e times (s), first {n_plot}: {np.round(tau_e[:n_plot], 3)}")


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\acf5s\step3\acf.png
K pairs_FM1 1/e times (s), first 10: [1.338 1.471 0.867 0.872 0.668 0.608 0.693 0.717 0.065 0.121]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM2\acf5s\step3\acf.png
K pairs_FM2 1/e times (s), first 10: [1.101 1.132 0.78  0.684 0.623 0.598 0.704 0.55  0.124 0.107]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_M1M2\acf5s\step3\acf.png
K pairs_M1M2 1/e times (s), first 10: [1.238 1.224 0.691 0.663 0.572 0.557 0.651 0.579 0.106 0.103]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\triple\acf5s\step3\acf.png
K triple 1/e times (s), first 10: [1.234 1.222 1.29  1.283 1.238 1.117 1.108 1.126 1.029 0.746]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\pairs_FM1\acf5s\step3\acf.png
P pairs_FM1 1/e times (s), first 10: [1.445 1.307 0.771 0.752 0.969 0.802 0.968 0.784 0.133 0.191]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\pairs_FM2\acf5s\step3\acf.png
P pairs_FM2 1/e times (s), first 10: [1.391 1.423 0.918 0.903 0.974 0.855 1.047 0.868 0.116 0.185]


... (2 outputs left out)

 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\pairs_FM1\acf5s\step3\acf.png
KP pairs_FM1 1/e times (s), first 10: [1.359 1.409 0.788 0.8   0.83  0.724 0.805 0.754 0.115 0.148]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\pairs_FM2\acf5s\step3\acf.png
KP pairs_FM2 1/e times (s), first 10: [1.251 1.291 0.882 0.755 0.783 0.723 0.825 0.697 0.132 0.14 ]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\pairs_M1M2\acf5s\step3\acf.png
KP pairs_M1M2 1/e times (s), first 10: [1.343 1.411 0.851 0.787 0.708 0.646 0.763 0.664 0.118 0.107]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\acf5s\step3\acf.png
KP triple 1/e times (s), first 10: [1.268 1.294 1.312 1.345 1.265 1.231 1.254 1.249 1.186 0.808]


## Step 4 — Discretise

Bin edges run from the minimum to the maximum inside the current analysis set, in equal widths. Both τ values use these edges. `joint` combines the selected components into one state. `per_pc` bins each component alone.

`STATE_VARS = "pcs"` bins `PCS_FOR_STATE`. `STATE_VARS = "physical"` is only for the pairs: `D_PP` on `[0, D_MAX]` and two angles on `[-pi, pi]`. Frames with `D_PP` above `D_MAX` are left out of the state, and the fraction is printed. The angle bins are circular: the first and the last are neighbours. The output tag contains `vars-pcs` or `vars-physical`.


In [7]:
pc_index = [p - 1 for p in PCS_FOR_STATE]


def build_physical(analysis, version, pair_name, fish_names, item):
    # 版本 A 的三个物理量。角度是环形的，-pi 与 pi 相邻；分格编号本身不把两端粘在一起。
    tag = state_tag(version)
    out = loc(analysis, version, tag, "step4")
    d_edges = np.linspace(0.0, float(D_MAX), int(N_BINS_D) + 1)
    th_edges = np.linspace(-np.pi, np.pi, int(N_BINS_THETA) + 1)
    widths = [int(N_BINS_D), int(N_BINS_THETA), int(N_BINS_THETA)]
    labels = ["D_PP", "theta_1", "theta_2"]
    item["state_mode"] = "physical"
    item["state_labels"] = labels
    item["state_widths"] = widths
    np.savez(out / "bin_edges.npz", D_PP=d_edges, theta_1=th_edges, theta_2=th_edges)
    print(f"\n{analysis} {version} physical bins  D 0..{D_MAX} {unit_name()} ({N_BINS_D})  "
          f"theta -pi..pi ({N_BINS_THETA})  order ({fish_names[0]}, {fish_names[1]})")
    print("  positive theta is counterclockwise from the pec-pec line to the pec-head line")
    by_name = {v["experiment"]: v for v in videos_for(analysis)}
    packed = []
    n_over = 0
    n_finite = 0
    for video in item["videos"]:
        src = by_name[video["experiment"]]
        min_xy = MIN_XY_MM
        if NORM_BL:
            min_xy = MIN_XY_MM / float(np.median(src["body_length"]))
        dist, th1, th2 = physical_quantities(src["nodes"], pair_name, min_xy)
        finite = np.isfinite(dist)
        n_finite += int(finite.sum())
        n_over += int((finite & (dist > D_MAX)).sum())
        cols = [digitize(dist, d_edges), digitize(th1, th_edges), digitize(th2, th_edges)]
        values = np.column_stack([dist, th1, th2])
        base = {
            "experiment": video["experiment"], "frames": video["frames"],
            "seg": video["seg"], "fps": video["fps"],
            "cols": cols, "values": values, "swapped": False,
        }
        packed.append(base)
        if version == "pairs_M1M2" and SYMMETRIZE_M1M2:
            packed.append({
                "experiment": video["experiment"], "frames": video["frames"],
                "seg": video["seg"], "fps": video["fps"],
                "cols": [cols[0], cols[2], cols[1]],
                "values": np.column_stack([dist, th2, th1]),
                "swapped": True,
            })
    if n_finite:
        print(f"  D_PP > D_MAX: {n_over}/{n_finite} = {n_over / n_finite:.4%}")
    else:
        print("  D_PP > D_MAX: no finite distances")
    modes = []
    if DO_JOINT:
        modes.append("joint")
    if DO_PER_PC:
        modes.append("per_pc")
    for mode in modes:
        frames = []
        item.setdefault("states", {})[mode] = {
            "n_states": int(np.prod(widths)) if mode == "joint" else int(max(widths)),
            "per_video": [],
            "labels": labels,
            "n_bins": widths,
        }
        for block in packed:
            row = {
                "experiment": block["experiment"],
                "frame": block["frames"],
                "swapped": int(block["swapped"]),
            }
            for lab, col in zip(labels, block["cols"]):
                row[f"{lab}_bin"] = col
            entry = {
                "seg": block["seg"], "fps": block["fps"],
                "experiment": block["experiment"], "frames": block["frames"],
                "values": block["values"], "swapped": block["swapped"],
            }
            if mode == "joint":
                state = joint_from_bins(block["cols"], widths)
                row["state"] = state
                entry["state"] = state
            else:
                entry["bins"] = block["cols"]
            item["states"][mode]["per_video"].append(entry)
            frames.append(pd.DataFrame(row))
        table = pd.concat(frames, ignore_index=True)
        table.to_csv(out / f"microstates_{mode}.csv", index=False)
        if mode == "joint":
            states = np.concatenate([
                v["state"] for v in item["states"][mode]["per_video"] if not v["swapped"]
            ])
            counts = occupancy(states, item["states"][mode]["n_states"])
            occupied = int((counts > 0).sum())
            print(f"  joint occupied {occupied} / {len(counts)}")
            order = np.argsort(counts)[::-1]
            order = order[counts[order] > 0]
            fig, ax = plt.subplots(figsize=(8, 3.5))
            ax.bar(np.arange(len(order)), counts[order], width=1.0, color="0.25")
            ax.set_xlabel("occupied state, sorted by count")
            ax.set_ylabel("frames")
            ax.set_title(f"{analysis} {version} joint  {occupied}/{len(counts)} states occupied\n"
                         "theta is circular: -pi and +pi bins are neighbours")
            save_fig(fig, out / "occupancy_joint.png")
        else:
            fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
            for ax, j, lab, n_bins in zip(axes, range(3), labels, widths):
                states = np.concatenate([
                    v["bins"][j] for v in item["states"][mode]["per_video"] if not v["swapped"]
                ])
                counts = occupancy(states, n_bins)
                occupied = int((counts > 0).sum())
                ax.bar(np.arange(n_bins), counts, color="0.25")
                ax.set_xlabel("bin")
                title = f"{lab}  {occupied}/{n_bins}"
                if lab.startswith("theta"):
                    title += "\n-pi and +pi bins are neighbours"
                ax.set_title(title)
                print(f"  per-variable {lab} occupied {occupied} / {n_bins}")
            axes[0].set_ylabel("frames")
            fig.suptitle(f"{analysis} {version} physical variables")
            fig.tight_layout()
            save_fig(fig, out / "occupancy_per_pc.png")


for analysis in SETS:
    for version, pair_name, fish_names in list(PAIRS) + [("triple", None, FISH)]:
        item = STORE[(analysis, version)]
        if STATE_VARS == "physical" and version != "triple":
            build_physical(analysis, version, pair_name, fish_names, item)
            continue
        if STATE_VARS == "physical" and version == "triple":
            print(f"\n{analysis} triple stays on PC bins; physical variables are defined for pairs")
        item["state_mode"] = "pcs"
        item["state_labels"] = [f"PC{pc}" for pc in PCS_FOR_STATE]
        item["state_widths"] = [N_BINS] * len(PCS_FOR_STATE)
        tag = state_tag(version)
        stacked = np.concatenate([v["scores"][:, pc_index] for v in item["videos"]], axis=0)
        edges = np.vstack([bin_edges(stacked[:, j]) for j in range(stacked.shape[1])])
        item["edges"] = edges
        print(f"\n{analysis} {version} bin edges")
        for j, pc in enumerate(PCS_FOR_STATE):
            print(f"  PC{pc}: {edges[j, 0]:.5f} .. {edges[j, -1]:.5f}  width {(edges[j, 1] - edges[j, 0]):.5f}")
        out = loc(analysis, version, tag, "step4")
        np.save(out / "bin_edges.npy", edges)
        modes = []
        if DO_JOINT:
            modes.append("joint")
        if DO_PER_PC:
            modes.append("per_pc")
        for mode in modes:
            frames = []
            state_blocks = []
            bin_blocks = []
            for video in item["videos"]:
                cols = [digitize(video["scores"][:, pc_index[j]], edges[j]) for j in range(len(pc_index))]
                if mode == "joint":
                    state = joint_from_bins(cols)
                    n_states = N_BINS ** len(pc_index)
                else:
                    state = None
                    n_states = N_BINS
                for j, col in enumerate(cols):
                    pass
                block = pd.DataFrame({
                    "experiment": video["experiment"],
                    "frame": video["frames"],
                })
                for j, pc in enumerate(PCS_FOR_STATE):
                    block[f"pc{pc}_bin"] = cols[j]
                if mode == "joint":
                    block["state"] = state
                    state_blocks.append(state)
                else:
                    for j, pc in enumerate(PCS_FOR_STATE):
                        block[f"pc{pc}_state"] = cols[j]
                frames.append(block)
                bin_blocks.append(cols)
            table = pd.concat(frames, ignore_index=True)
            table.to_csv(out / f"microstates_{mode}.csv", index=False)
            item.setdefault("states", {})[mode] = {
                "n_states": N_BINS ** len(pc_index) if mode == "joint" else N_BINS,
                "per_video": [],
                "labels": item["state_labels"],
                "n_bins": item["state_widths"],
            }
            for video, cols in zip(item["videos"], bin_blocks):
                extra = {
                    "seg": video["seg"], "fps": video["fps"],
                    "experiment": video["experiment"], "frames": video["frames"],
                    "values": video["scores"][:, pc_index], "swapped": False,
                }
                if mode == "joint":
                    state = joint_from_bins(cols)
                    extra["state"] = state
                    item["states"][mode]["per_video"].append(extra)
                else:
                    extra["bins"] = cols
                    item["states"][mode]["per_video"].append(extra)
            if mode == "joint":
                states = np.concatenate([v["state"] for v in item["states"][mode]["per_video"]])
                counts = occupancy(states, item["states"][mode]["n_states"])
                occupied = int((counts > 0).sum())
                print(f"  joint occupied {occupied} / {len(counts)}")
                order = np.argsort(counts)[::-1]
                order = order[counts[order] > 0]
                fig, ax = plt.subplots(figsize=(8, 3.5))
                ax.bar(np.arange(len(order)), counts[order], width=1.0, color="0.25")
                ax.set_xlabel("occupied state, sorted by count")
                ax.set_ylabel("frames")
                ax.set_title(f"{analysis} {version} joint  {occupied}/{len(counts)} states occupied")
                save_fig(fig, out / "occupancy_joint.png")
            else:
                fig, axes = plt.subplots(1, len(pc_index), figsize=(4 * len(pc_index), 3.2))
                for ax, j, pc in zip(np.atleast_1d(axes), range(len(pc_index)), PCS_FOR_STATE):
                    states = np.concatenate([v["bins"][j] for v in item["states"][mode]["per_video"]])
                    counts = occupancy(states, N_BINS)
                    occupied = int((counts > 0).sum())
                    ax.bar(np.arange(N_BINS), counts, color="0.25")
                    ax.set_xlabel("bin")
                    ax.set_title(f"PC{pc}  {occupied}/{N_BINS}")
                    print(f"  per-pc PC{pc} occupied {occupied} / {N_BINS}")
                axes[0].set_ylabel("frames")
                fig.suptitle(f"{analysis} {version} per PC")
                fig.tight_layout()
                save_fig(fig, out / "occupancy_per_pc.png")



K pairs_FM1 bin edges
  PC1: -263.81430 .. 168.39929  width 43.22136
  PC2: -143.71357 .. 167.54684  width 31.12604
  PC3: -29.27542 .. 28.87977  width 5.81552


  joint occupied 867 / 1000


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs\step4\occupancy_joint.png


  per-pc PC1 occupied 10 / 10
  per-pc PC2 occupied 10 / 10
  per-pc PC3 occupied 10 / 10
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs\step4\occupancy_per_pc.png

K pairs_FM2 bin edges
  PC1: -160.19657 .. 247.09190  width 40.72885
  PC2: -238.04373 .. 184.23551  width 42.22792
  PC3: -34.09420 .. 31.74909  width 6.58433


  joint occupied 580 / 1000


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM2\bins10_pcs1-2-3_vars-pcs\step4\occupancy_joint.png


... (27 outputs left out)

  per-pc PC1 occupied 10 / 10
  per-pc PC2 occupied 10 / 10
  per-pc PC3 occupied 8 / 10
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\pairs_M1M2\bins10_pcs1-2-3_vars-pcs\step4\occupancy_per_pc.png

KP triple bin edges
  PC1: -318.02619 .. 352.68080  width 67.07070
  PC2: -341.02457 .. 283.52486  width 62.45494
  PC3: -282.95959 .. 334.38909  width 61.73487


  joint occupied 809 / 1000


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs\step4\occupancy_joint.png


  per-pc PC1 occupied 10 / 10
  per-pc PC2 occupied 10 / 10
  per-pc PC3 occupied 10 / 10
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs\step4\occupancy_per_pc.png


## Step 5 — Transition matrices

A pair of frames counts only when both states are valid and both frames sit in the same continuous segment. Rows with no counts stay zero. Joint heatmaps use a log scale.


In [8]:
def draw_heatmap(prob, path, title, log_scale):
    dense = prob.toarray()
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    if log_scale:
        show = np.ma.masked_where(dense <= 0, dense)
        positive = dense[dense > 0]
        if len(positive) == 0:
            ax.set_title(title + "\nno counts")
            save_fig(fig, path)
            return
        im = ax.imshow(show, norm=LogNorm(vmin=max(float(positive.min()), 1e-8), vmax=float(positive.max())),
                       cmap="viridis", interpolation="nearest")
    else:
        im = ax.imshow(dense, vmin=0, vmax=1, cmap="viridis", interpolation="nearest")
    fig.colorbar(im, ax=ax, fraction=0.046, label="transition probability")
    ax.set_xlabel("state at t+tau")
    ax.set_ylabel("state at t")
    ax.set_title(title)
    save_fig(fig, path)


def one_tau(item, mode, tau):
    n_states = item["states"][mode]["n_states"]
    counts = sparse.csr_matrix((n_states, n_states), dtype=np.float64)
    n_pairs = 0
    fps_values = []
    if mode == "joint":
        for video in item["states"][mode]["per_video"]:
            part, n = transition_matrix(video["state"], video["seg"], tau, n_states)
            counts = counts + part
            n_pairs += n
            fps_values.append(video["fps"])
    else:
        # Caller passes one PC at a time by temporarily wrapping bins as state.
        raise RuntimeError("use per-pc wrapper")
    return counts, n_pairs, fps_values


for analysis in SETS:
    for version, _pair, _fish in list(PAIRS) + [("triple", None, FISH)]:
        item = STORE[(analysis, version)]
        tag_base = state_tag(version)
        for tau in TAUS:
            if DO_JOINT:
                counts, n_pairs, fps_values = one_tau(item, "joint", tau)
                prob, row_sum = row_normalize(counts)
                prob_sum = np.asarray(prob.sum(axis=1)).ravel()
                occupied = row_sum > 0
                max_err = float(np.max(np.abs(prob_sum[occupied] - 1))) if occupied.any() else np.nan
                stay = stay_fraction(counts)
                seconds = [tau / fps for fps in sorted(set(np.round(fps_values, 5)))]
                print(f"{analysis} {version} joint tau={tau} frames ({seconds} s)  "
                      f"pairs {n_pairs}  row-sum error {max_err:.3e}  stay {stay:.4f}")
                tag = f"{tag_base}_tau{tau}_joint"
                out = loc(analysis, version, tag, "step5")
                sparse.save_npz(out / "counts.npz", counts)
                sparse.save_npz(out / "probability.npz", prob)
                pd.DataFrame(counts.toarray()).to_csv(out / "counts.csv", index=False)
                pd.DataFrame(prob.toarray()).to_csv(out / "probability.csv", index=False)
                joint_title = (f"{analysis} {version} joint  tau={tau} frames ({seconds[0]:.4f} s)\n"
                               "log scale, empty cells masked")
                if item.get("state_mode") == "physical":
                    joint_title += "\ntheta bins are circular: -pi and +pi are neighbours"
                draw_heatmap(prob, out / "transition.png", joint_title, log_scale=True)
                pd.DataFrame([{
                    "pairs": n_pairs, "max_row_sum_error": max_err, "stay_probability": stay,
                    "tau_frames": tau, "tau_seconds": seconds[0],
                }]).to_csv(out / "checks.csv", index=False)
            if DO_PER_PC:
                labels = item["states"]["per_pc"].get("labels", [f"PC{pc}" for pc in PCS_FOR_STATE])
                n_bins_list = item["states"]["per_pc"].get("n_bins", [N_BINS] * len(labels))
                fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3.6))
                check_rows = []
                tag = f"{tag_base}_tau{tau}_perpc"
                out = loc(analysis, version, tag, "step5")
                for j, label in enumerate(labels):
                    n_bins = int(n_bins_list[j])
                    counts = sparse.csr_matrix((n_bins, n_bins), dtype=np.float64)
                    n_pairs = 0
                    fps_values = []
                    for video in item["states"]["per_pc"]["per_video"]:
                        part, n = transition_matrix(video["bins"][j], video["seg"], tau, n_bins)
                        counts = counts + part
                        n_pairs += n
                        fps_values.append(video["fps"])
                    prob, row_sum = row_normalize(counts)
                    prob_sum = np.asarray(prob.sum(axis=1)).ravel()
                    occupied = row_sum > 0
                    max_err = float(np.max(np.abs(prob_sum[occupied] - 1))) if occupied.any() else np.nan
                    stay = stay_fraction(counts)
                    seconds = tau / float(fps_values[0])
                    print(f"  per-variable {label} tau={tau} pairs {n_pairs} row-sum error {max_err:.3e} stay {stay:.4f}")
                    stem = label if item.get("state_mode") == "physical" else f"pc{PCS_FOR_STATE[j]}"
                    sparse.save_npz(out / f"{stem}_counts.npz", counts)
                    sparse.save_npz(out / f"{stem}_probability.npz", prob)
                    pd.DataFrame(counts.toarray()).to_csv(out / f"{stem}_counts.csv", index=False)
                    pd.DataFrame(prob.toarray()).to_csv(out / f"{stem}_probability.csv", index=False)
                    ax = np.atleast_1d(axes)[j]
                    dense = prob.toarray()
                    im = ax.imshow(dense, vmin=0, vmax=1, cmap="viridis")
                    ax.set_title(f"{label} stay {stay:.2f}")
                    ax.set_xlabel("bin t+tau")
                    ax.set_ylabel("bin t")
                    check_rows.append({
                        "pc": None if item.get("state_mode") == "physical" else int(PCS_FOR_STATE[j]),
                        "variable": label, "pairs": n_pairs, "max_row_sum_error": max_err,
                        "stay_probability": stay, "tau_frames": tau, "tau_seconds": seconds,
                    })
                fig.colorbar(im, ax=list(np.atleast_1d(axes)), fraction=0.046, label="probability")
                if item.get("state_mode") == "physical":
                    sup = (f"{analysis} {version} per variable  tau={tau} frames ({seconds:.4f} s)\n"
                           "theta bins are circular: -pi and +pi are neighbours")
                else:
                    sup = f"{analysis} {version} per PC  tau={tau} frames ({seconds:.4f} s)"
                fig.suptitle(sup)
                fig.subplots_adjust(top=0.78, wspace=0.35)
                save_fig(fig, out / "transition.png")
                pd.DataFrame(check_rows).to_csv(out / "checks.csv", index=False)


K pairs_FM1 joint tau=7 frames ([np.float64(0.04216867469879518)] s)  pairs 651438  row-sum error 3.331e-16  stay 0.7219


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau7_joint\step5\transition.png
  per-variable PC1 tau=7 pairs 651438 row-sum error 2.220e-16 stay 0.9560
  per-variable PC2 tau=7 pairs 651438 row-sum error 1.110e-16 stay 0.9464
  per-variable PC3 tau=7 pairs 651438 row-sum error 2.220e-16 stay 0.7924


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau7_perpc\step5\transition.png
K pairs_FM1 joint tau=17 frames ([np.float64(0.10240963855421686)] s)  pairs 644131  row-sum error 3.331e-16  stay 0.5364


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_joint\step5\transition.png
  per-variable PC1 tau=17 pairs 644131 row-sum error 0.000e+00 stay 0.8954
  per-variable PC2 tau=17 pairs 644131 row-sum error 1.110e-16 stay 0.8840
  per-variable PC3 tau=17 pairs 644131 row-sum error 2.220e-16 stay 0.6615


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_perpc\step5\transition.png
K pairs_FM2 joint tau=7 frames ([np.float64(0.04216867469879518)] s)  pairs 649226  row-sum error 3.331e-16  stay 0.7606


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM2\bins10_pcs1-2-3_vars-pcs_tau7_joint\step5\transition.png
  per-variable PC1 tau=7 pairs 649226 row-sum error 1.110e-16 stay 0.9614
  per-variable PC2 tau=7 pairs 649226 row-sum error 1.110e-16 stay 0.9643
  per-variable PC3 tau=7 pairs 649226 row-sum error 2.220e-16 stay 0.8155


... (41 outputs left out)

 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs_tau7_joint\step5\transition.png
  per-variable PC1 tau=7 pairs 1283662 row-sum error 1.110e-16 stay 0.9662
  per-variable PC2 tau=7 pairs 1283662 row-sum error 1.110e-16 stay 0.9682
  per-variable PC3 tau=7 pairs 1283662 row-sum error 1.110e-16 stay 0.9659


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs_tau7_perpc\step5\transition.png
KP triple joint tau=17 frames ([np.float64(0.10240963855421686)] s)  pairs 1263619  row-sum error 3.331e-16  stay 0.7923


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs_tau17_joint\step5\transition.png
  per-variable PC1 tau=17 pairs 1263619 row-sum error 1.110e-16 stay 0.9192
  per-variable PC2 tau=17 pairs 1263619 row-sum error 1.110e-16 stay 0.9277
  per-variable PC3 tau=17 pairs 1263619 row-sum error 2.220e-16 stay 0.9224


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs_tau17_perpc\step5\transition.png


## Step 7 — Infomap on the joint transition matrix

This step reads the joint counts from Step 5. It does not read behaviour labels, and it does not run on the per-component matrices. One backend is used for the τ scan and for the figures. If neither python-igraph nor the infomap package imports, the notebook stops and prints an install command.


In [9]:
import random

# 本步骤只读 joint 转移计数，不读手动标记。两个后端都没有就停，不跳过。


def resolve_infomap_backend():
    try:
        import igraph
        INFOMAP_INFO.clear()
        INFOMAP_INFO.update({
            "backend": "python-igraph",
            "version": igraph.__version__,
            "call": "Graph.community_infomap(edge_weights=..., trials=N_TRIALS)",
            "seed": int(INFOMAP_SEED),
            "seed_argument": False,
        })
        print(f"Infomap backend python-igraph {igraph.__version__}. "
              f"community_infomap has no seed argument; random.seed({INFOMAP_SEED}) is set before each call.")
        return "igraph", igraph
    except ImportError:
        pass
    try:
        import infomap
        version = getattr(infomap, "__version__", "unknown")
        INFOMAP_INFO.clear()
        INFOMAP_INFO.update({
            "backend": "infomap",
            "version": version,
            "call": "Infomap(directed=True, num_trials=N_TRIALS)",
            "seed": int(INFOMAP_SEED),
            "seed_argument": True,
        })
        print(f"Infomap backend infomap {version}")
        return "infomap", infomap
    except ImportError:
        message = (
            "Infomap backend is not installed. Install one of:\n"
            r"  .venv\Scripts\python.exe -m pip install python-igraph" "\n"
            r"  .venv\Scripts\python.exe -m pip install infomap"
        )
        print(message)
        raise RuntimeError(message)


BACKEND_KIND, BACKEND_MODULE = resolve_infomap_backend()


def prepare_graph_counts(counts):
    # 去掉行和列都是 0 的状态。边权是转移概率。
    counts = counts.tocsr().astype(np.float64)
    row_sum = np.asarray(counts.sum(axis=1)).ravel()
    col_sum = np.asarray(counts.sum(axis=0)).ravel()
    keep = np.flatnonzero((row_sum > 0) | (col_sum > 0))
    if len(keep) == 0:
        raise RuntimeError("joint transition matrix has no occupied states")
    sub = counts[keep][:, keep].tocsr()
    if not KEEP_SELF_LOOPS:
        sub = sub.tolil()
        sub.setdiag(0)
        sub = sub.tocsr()
        sub.eliminate_zeros()
    prob, _row = row_normalize(sub)
    return keep, sub, prob


def run_infomap(prob, trials):
    coo = prob.tocoo()
    mask = coo.data > 0
    edges = list(zip(coo.row[mask].astype(int).tolist(), coo.col[mask].astype(int).tolist()))
    weights = coo.data[mask].astype(float).tolist()
    n = prob.shape[0]
    if BACKEND_KIND == "igraph":
        random.seed(INFOMAP_SEED)
        np.random.seed(INFOMAP_SEED)
        graph = BACKEND_MODULE.Graph(n=n, edges=edges, directed=True)
        if weights:
            graph.es["weight"] = weights
            clustering = graph.community_infomap(edge_weights="weight", trials=int(trials))
        else:
            clustering = graph.community_infomap(trials=int(trials))
        labels = np.asarray(clustering.membership, dtype=np.int32)
        codelength = float(getattr(clustering, "codelength", np.nan))
        return labels, codelength
    try:
        model = BACKEND_MODULE.Infomap(directed=True, num_trials=int(trials), seed=int(INFOMAP_SEED), silent=True)
    except TypeError:
        model = BACKEND_MODULE.Infomap(directed=True, num_trials=int(trials), silent=True)
    for src, dst, weight in zip(coo.row[mask], coo.col[mask], coo.data[mask]):
        model.add_link(int(src), int(dst), float(weight))
    model.run()
    labels = np.full(n, -1, dtype=np.int32)
    modules = model.get_modules() if hasattr(model, "get_modules") else {}
    for node, module in modules.items():
        labels[int(node)] = int(module)
    missing = np.flatnonzero(labels < 0)
    start = int(labels.max()) + 1 if np.any(labels >= 0) else 0
    for k, node in enumerate(missing):
        labels[node] = start + k
    return labels, np.nan


def weighted_within(counts, labels):
    coo = counts.tocoo()
    if coo.nnz == 0 or coo.data.sum() <= 0:
        return np.nan
    same = labels[coo.row] == labels[coo.col]
    return float(coo.data[same].sum() / coo.data.sum())


def within_rows(counts, labels):
    rows = []
    for cluster in sorted(set(labels.tolist())):
        idx = np.flatnonzero(labels == cluster)
        inside = float(counts[idx][:, idx].sum())
        outgoing = float(counts[idx].sum())
        rows.append({
            "cluster": int(cluster),
            "n_states": int(len(idx)),
            "within_mass": inside,
            "outgoing_mass": outgoing,
            "within_fraction": inside / outgoing if outgoing else np.nan,
        })
    return rows


def cluster_frame_labels(item, cluster_of_state):
    labelled = []
    for block in item["states"]["joint"]["per_video"]:
        state = block["state"]
        lab = np.full(len(state), -1, dtype=np.int32)
        ok = state >= 0
        lab[ok] = cluster_of_state[state[ok]]
        labelled.append((block, lab))
    return labelled


def dwell_lengths(lab, seg):
    out = {}
    n = len(lab)
    i = 0
    while i < n:
        if lab[i] < 0 or seg[i] < 0:
            i += 1
            continue
        j = i + 1
        cid = int(lab[i])
        sid = seg[i]
        while j < n and lab[j] == cid and seg[j] == sid:
            j += 1
        out.setdefault(cid, []).append(j - i)
        i = j
    return out


def select_clusters(frame_count, n_valid):
    ordered = sorted(frame_count, key=lambda c: (-frame_count[c], c))
    if CLUSTER_SELECT == "all":
        selected = ordered
    elif CLUSTER_SELECT == "top_coverage":
        selected = []
        acc = 0
        for cluster in ordered:
            selected.append(cluster)
            acc += frame_count[cluster]
            if n_valid > 0 and acc / n_valid >= COVERAGE:
                break
    else:
        raise RuntimeError(f"CLUSTER_SELECT must be 'all' or 'top_coverage', got {CLUSTER_SELECT!r}")
    covered = sum(frame_count[c] for c in selected)
    return selected, covered


def analyse_joint(item, tau, trials):
    counts, n_pairs, fps_values = one_tau(item, "joint", tau)
    n_states = item["states"]["joint"]["n_states"]
    keep, sub, prob = prepare_graph_counts(counts)
    raw_labels, codelength = run_infomap(prob, trials)
    # 先用原始编号数帧，再按覆盖帧数把簇重编号，0 是覆盖最多的簇。
    raw_of_state = np.full(n_states, -1, dtype=np.int32)
    raw_of_state[keep] = raw_labels
    frame_count = {}
    n_valid = 0
    for block in item["states"]["joint"]["per_video"]:
        if block["swapped"]:
            continue
        state = block["state"]
        ok = state >= 0
        n_valid += int(ok.sum())
        ids = raw_of_state[state[ok]]
        present = ids[ids >= 0]
        if len(present):
            bc = np.bincount(present)
            for cid, n in enumerate(bc):
                if n:
                    frame_count[int(cid)] = frame_count.get(int(cid), 0) + int(n)
    for cid in sorted(set(raw_labels.tolist())):
        frame_count.setdefault(int(cid), 0)
    ordered = sorted(frame_count, key=lambda c: (-frame_count[c], c))
    remap = {old: new for new, old in enumerate(ordered)}
    labels = np.array([remap[int(c)] for c in raw_labels], dtype=np.int32)
    cluster_of_state = np.full(n_states, -1, dtype=np.int32)
    cluster_of_state[keep] = labels
    frame_count = {remap[c]: frame_count[c] for c in frame_count}
    labelled = cluster_frame_labels(item, cluster_of_state)
    selected, covered = select_clusters(frame_count, n_valid)
    sizes = pd.Series(labels).value_counts()
    largest_states = int(sizes.max()) if len(sizes) else 0
    volume = largest_states / n_states if n_states else np.nan
    within = weighted_within(sub, labels)
    no_diag = sub.copy().tolil()
    no_diag.setdiag(0)
    no_diag = no_diag.tocsr()
    no_diag.eliminate_zeros()
    within_no_diag = weighted_within(no_diag, labels)
    return {
        "counts": counts, "n_pairs": n_pairs, "fps_values": fps_values,
        "n_states": n_states, "keep": keep, "sub": sub, "prob": prob,
        "labels": labels, "codelength": codelength,
        "cluster_of_state": cluster_of_state, "frame_count": frame_count,
        "n_valid": n_valid, "selected": selected, "covered": covered,
        "n_clusters": int(len(sizes)), "largest_states": largest_states,
        "volume_fraction": volume, "within": within, "within_no_diag": within_no_diag,
        "no_diag": no_diag, "labelled": labelled,
    }


def draw_block(result, analysis, version, tau, path):
    selected = result["selected"]
    labels = result["labels"]
    keep = result["keep"]
    members = {int(c): [] for c in selected}
    for local, state in enumerate(keep):
        cluster = int(labels[local])
        if cluster in members:
            members[cluster].append(int(state))
    order = []
    for cluster in selected:
        order.extend(sorted(members[cluster]))
    if not order:
        raise RuntimeError(f"{analysis} {version} tau={tau}: no states selected")
    prob = result["counts"].tocsr().astype(np.float64)
    if not KEEP_SELF_LOOPS:
        prob = prob.tolil()
        prob.setdiag(0)
        prob = prob.tocsr()
    prob, _row = row_normalize(prob)
    block = prob[np.ix_(order, order)]
    sub = np.asarray(block.toarray() if sparse.issparse(block) else block, dtype=np.float64)
    positive = sub[sub > 0]
    fig, ax = plt.subplots(figsize=(8.2, 7.2))
    if len(positive) == 0:
        ax.set_title("no positive probabilities")
        save_fig(fig, path)
        return sub, order
    vmin = float(positive.min())
    vmax = float(positive.max())
    if vmax <= vmin:
        vmax = vmin * 10.0 if vmin > 0 else 1.0
    show = np.ma.masked_where(sub <= 0, sub)
    im = ax.imshow(show, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="viridis", interpolation="nearest")
    fig.colorbar(im, ax=ax, fraction=0.046, label="transition probability")
    sizes = [len(members[c]) for c in selected]
    cuts = np.cumsum(sizes)[:-1] - 0.5
    if len(cuts):
        ax.hlines(cuts, -0.5, len(order) - 0.5, colors="k", linewidth=0.4)
        ax.vlines(cuts, -0.5, len(order) - 0.5, colors="k", linewidth=0.4)
    if len(selected) <= 24:
        start = 0
        for cluster, n in zip(selected, sizes):
            mid = start + (n - 1) / 2.0
            ax.text(mid, -1.2, f"{cluster}\n(n={n})", ha="center", va="top", fontsize=6, clip_on=False)
            start += n
    fps = float(np.median(result["fps_values"])) if result["fps_values"] else np.nan
    ax.set_xlabel("state at t+tau")
    ax.set_ylabel("state at t")
    ax.set_title(
        f"{analysis} {version}  tau={tau} frames ({tau / fps:.4f} s)\n"
        f"{len(order)} states, {len(selected)} clusters, "
        f"{INFOMAP_INFO['backend']} {INFOMAP_INFO['version']}, trials={N_TRIALS}"
    )
    fig.tight_layout()
    save_fig(fig, path)
    return sub, order


def shuffle_within(counts, labels, tau):
    rng = np.random.default_rng(int(INFOMAP_SEED) + int(tau))
    values = np.empty(N_SHUFFLE, dtype=np.float64)
    for i in range(N_SHUFFLE):
        values[i] = weighted_within(counts, rng.permutation(labels))
    return float(np.nanmean(values)), float(np.nanmin(values)), float(np.nanmax(values))


def cluster_table(result, labels_3):
    rows = []
    dwell = {int(c): [] for c in result["frame_count"]}
    dwell_sec = {int(c): [] for c in result["frame_count"]}
    value_parts = {int(c): [] for c in result["frame_count"]}
    for block, lab in result["labelled"]:
        if block["swapped"]:
            continue
        runs = dwell_lengths(lab, block["seg"])
        for cluster, lengths in runs.items():
            dwell.setdefault(cluster, []).extend(lengths)
            dwell_sec.setdefault(cluster, []).extend(length / block["fps"] for length in lengths)
        for cluster in result["frame_count"]:
            mask = lab == cluster
            if np.any(mask):
                value_parts.setdefault(cluster, []).append(block["values"][mask])
    selected = set(result["selected"])
    for cluster, n_frames in sorted(result["frame_count"].items()):
        runs = dwell.get(cluster, [])
        secs = dwell_sec.get(cluster, [])
        row = {
            "cluster": int(cluster),
            "selected": int(cluster in selected),
            "n_states": int(np.sum(result["labels"] == cluster)),
            "n_frames": int(n_frames),
            "mean_dwell_frames": float(np.mean(runs)) if runs else np.nan,
            "mean_dwell_seconds": float(np.mean(secs)) if secs else np.nan,
        }
        if value_parts.get(cluster):
            values = np.concatenate(value_parts[cluster], axis=0)
            for j, name in enumerate(labels_3):
                col = values[:, j]
                finite = col[np.isfinite(col)]
                row[f"{name}_min"] = float(finite.min()) if len(finite) else np.nan
                row[f"{name}_max"] = float(finite.max()) if len(finite) else np.nan
        else:
            for name in labels_3:
                row[f"{name}_min"] = np.nan
                row[f"{name}_max"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


if not DO_JOINT:
    raise RuntimeError("Infomap needs the joint transition matrix. DO_JOINT is False.")

scan_rows = []
for analysis in SETS:
    for version, _pair, _fish in list(PAIRS) + [("triple", None, FISH)]:
        item = STORE[(analysis, version)]
        if "joint" not in item.get("states", {}):
            raise RuntimeError(f"{analysis} {version} has no joint states")
        labels_3 = item.get("state_labels", [f"PC{pc}" for pc in PCS_FOR_STATE])
        print(f"\n== Infomap {analysis} {version}  full grid {item['states']['joint']['n_states']}  "
              f"backend {INFOMAP_INFO['backend']} {INFOMAP_INFO['version']}")
        for tau in TAU_SCAN:
            result = analyse_joint(item, tau, N_TRIALS_SCAN)
            scan_rows.append({
                "analysis": analysis,
                "version": version,
                "tau_frames": int(tau),
                "n_clusters": result["n_clusters"],
                "largest_cluster_volume_fraction": result["volume_fraction"],
                "n_occupied_states": int(len(result["keep"])),
                "n_pairs": int(result["n_pairs"]),
                "within_cluster_fraction": result["within"],
                "within_cluster_fraction_no_diagonal": result["within_no_diag"],
                "trials": int(N_TRIALS_SCAN),
            })
            print(f"  scan tau={tau} clusters {result['n_clusters']}  "
                  f"largest volume {result['volume_fraction']:.4f}  "
                  f"occupied {len(result['keep'])}  pairs {result['n_pairs']}  "
                  f"within {result['within']:.4f}")
        scan_df = pd.DataFrame(scan_rows)
        scan_df = scan_df[(scan_df["analysis"] == analysis) & (scan_df["version"] == version)]
        fig, ax = plt.subplots(figsize=(7.2, 4.2))
        ax.plot(scan_df["tau_frames"], scan_df["largest_cluster_volume_fraction"], marker="o", color="#4C78A8")
        ax.set_xlabel("tau (frames)")
        ax.set_ylabel("largest cluster states / full grid")
        inset = ax.inset_axes([0.58, 0.15, 0.38, 0.38])
        inset.plot(scan_df["tau_frames"], scan_df["n_clusters"], marker="o", color="0.25", ms=3)
        inset.set_xlabel("tau (frames)", fontsize=7)
        inset.set_ylabel("clusters", fontsize=7)
        inset.tick_params(labelsize=7)
        for mark in (7, 17):
            hit = scan_df.loc[scan_df["tau_frames"] == mark]
            if len(hit):
                y = float(hit["largest_cluster_volume_fraction"].iloc[0])
                ax.axvline(mark, color="0.6", ls="--", lw=0.8)
                ax.annotate(str(mark), (mark, y), textcoords="offset points", xytext=(4, 6), fontsize=8)
        ax.set_title(f"{analysis} {version}  volume fraction of the largest cluster\n"
                     f"{INFOMAP_INFO['backend']} {INFOMAP_INFO['version']}, trials={N_TRIALS_SCAN}")
        scan_dir = loc(analysis, version, f"{state_tag(version)}_tau_scan", "step7")
        save_fig(fig, scan_dir / "tau_scan.png")
        scan_df.to_csv(scan_dir / "tau_scan.csv", index=False)
        print(f"tau 7 and 17 on this scan ({N_TRIALS_SCAN} trials):")
        print(scan_df.loc[scan_df["tau_frames"].isin([7, 17])].to_string(index=False))

        for tau in TAUS:
            result = analyse_joint(item, tau, N_TRIALS)
            covered_frac = result["covered"] / result["n_valid"] if result["n_valid"] else np.nan
            print(f"\n{analysis} {version} tau={tau} trials={N_TRIALS}  "
                  f"clusters {result['n_clusters']}  selected {len(result['selected'])}  "
                  f"coverage {covered_frac:.4f}  codelength {result['codelength']:.4f}")
            print(f"  within-cluster fraction {result['within']:.4f}")
            if KEEP_SELF_LOOPS:
                print(f"  within-cluster fraction after removing the diagonal {result['within_no_diag']:.4f}")
            mean_s, min_s, max_s = shuffle_within(result["sub"], result["labels"], tau)
            print(f"  shuffled labels x{N_SHUFFLE}: mean {mean_s:.4f}  range {min_s:.4f} .. {max_s:.4f}")
            if KEEP_SELF_LOOPS:
                mean_d, min_d, max_d = shuffle_within(result["no_diag"], result["labels"], tau + 10000)
            else:
                mean_d = min_d = max_d = np.nan
            out = loc(analysis, version, f"{state_tag(version)}_tau{tau}_infomap", "step7")
            sub, order = draw_block(result, analysis, version, tau, out / "transition_blocks.png")
            cluster_of = np.full(result["n_states"], -1, dtype=np.int32)
            cluster_of[result["keep"]] = result["labels"]
            pd.DataFrame({
                "position": np.arange(len(order)),
                "state": order,
                "cluster": cluster_of[order],
            }).to_csv(out / "state_order.csv", index=False)
            np.savez(out / "probability_ordered.npz", probability=sub, state=np.asarray(order, dtype=np.int32),
                     cluster=cluster_of[np.asarray(order, dtype=np.int32)])
            state_rows = []
            for local, state in enumerate(result["keep"]):
                state_rows.append({
                    "state": int(state),
                    "cluster": int(result["labels"][local]),
                    "selected": int(int(result["labels"][local]) in set(result["selected"])),
                })
            pd.DataFrame(state_rows).to_csv(out / "cluster_states.csv", index=False)
            summary = cluster_table(result, labels_3)
            within_table = pd.DataFrame(within_rows(result["sub"], result["labels"]))
            summary = summary.merge(within_table[["cluster", "within_fraction", "within_mass", "outgoing_mass"]],
                                    on="cluster", how="left")
            if KEEP_SELF_LOOPS:
                no_diag_table = pd.DataFrame(within_rows(result["no_diag"], result["labels"]))
                no_diag_table = no_diag_table.rename(columns={"within_fraction": "within_fraction_no_diagonal"})
                summary = summary.merge(no_diag_table[["cluster", "within_fraction_no_diagonal"]],
                                        on="cluster", how="left")
            summary.to_csv(out / "cluster_summary.csv", index=False)
            print(summary.to_string(index=False))
            frame_parts = []
            for block, lab in result["labelled"]:
                if block["swapped"]:
                    continue
                frame_parts.append(pd.DataFrame({
                    "experiment": block["experiment"],
                    "frame": block["frames"],
                    "state": block["state"],
                    "cluster": lab,
                }))
            pd.concat(frame_parts, ignore_index=True).to_csv(out / "frame_clusters.csv", index=False)
            pd.DataFrame([{
                "tau_frames": int(tau),
                "tau_seconds": float(tau / np.median(result["fps_values"])),
                "backend": INFOMAP_INFO["backend"],
                "backend_version": INFOMAP_INFO["version"],
                "trials": int(N_TRIALS),
                "n_clusters": result["n_clusters"],
                "n_selected": len(result["selected"]),
                "coverage": covered_frac,
                "largest_cluster_volume_fraction": result["volume_fraction"],
                "n_occupied_states": int(len(result["keep"])),
                "n_pairs": int(result["n_pairs"]),
                "within_cluster_fraction": result["within"],
                "within_cluster_fraction_no_diagonal": result["within_no_diag"],
                "shuffle_mean": mean_s,
                "shuffle_min": min_s,
                "shuffle_max": max_s,
                "shuffle_no_diagonal_mean": mean_d,
                "shuffle_no_diagonal_min": min_d,
                "shuffle_no_diagonal_max": max_d,
                "codelength": result["codelength"],
                "keep_self_loops": bool(KEEP_SELF_LOOPS),
                "cluster_select": CLUSTER_SELECT,
                "coverage_target": float(COVERAGE),
            }]).to_csv(out / "checks.csv", index=False)

pd.DataFrame(scan_rows).to_csv(ROOT / "infomap_tau_scan.csv", index=False)
print("\nInfomap finished with", INFOMAP_INFO["backend"], INFOMAP_INFO["version"])


Infomap backend python-igraph 1.0.0. community_infomap has no seed argument; random.seed(20261106) is set before each call.

== Infomap K pairs_FM1  full grid 1000  backend python-igraph 1.0.0


  scan tau=1 clusters 739  largest volume 0.0040  occupied 867  pairs 657400  within 0.9428


  scan tau=2 clusters 545  largest volume 0.0060  occupied 867  pairs 656185  within 0.9246


  scan tau=3 clusters 440  largest volume 0.0080  occupied 867  pairs 655115  within 0.9079


  scan tau=5 clusters 357  largest volume 0.0100  occupied 866  pairs 653184  within 0.8786


  scan tau=7 clusters 305  largest volume 0.0110  occupied 866  pairs 651438  within 0.8531


... (230 outputs left out)


KP triple tau=17 trials=500  clusters 389  selected 313  coverage 0.9503  codelength 4.2168
  within-cluster fraction 0.8330
  within-cluster fraction after removing the diagonal 0.1959
  shuffled labels x100: mean 0.7928  range 0.7923 .. 0.7936


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\KP\triple\bins10_pcs1-2-3_vars-pcs_tau17_infomap\step7\transition_blocks.png


 cluster  selected  n_states  n_frames  mean_dwell_frames  mean_dwell_seconds     PC1_min     PC1_max     PC2_min     PC2_max     PC3_min     PC3_max  within_fraction  within_mass  outgoing_mass  within_fraction_no_diagonal
       0         1         4     12400          88.571429            0.533563  -49.739898   84.390391  158.616897  221.058923  -97.745192   25.714379         0.850802      10447.0        12279.0                     0.369146
       1         1         4     11543          75.940789            0.457475 -183.870504  -49.753540 -153.658272  -91.206035  -36.015805   87.449317         0.867405       9793.0        11290.0                     0.394172
       2         1         2     11270          96.324786            0.580270  151.473761  218.536787  -28.747430   33.693528  -36.017000   87.431850         0.869655       9701.0        11155.0                     0.230688
       3         1         2     10218         101.168317            0.609448   84.402373  151.458109 -1


Infomap finished with python-igraph 1.0.0


## Step 5b — extra transition figures

The block-diagonal figure stays in the Infomap output folder. The figures below are written under `transition_blocks/` in that same folder. They read the saved joint counts and the saved cluster files. They do not fit a new clustering.

Cluster numbers are already ordered by covered frames. Numbers in the cluster heatmap use two significant figures and leave out values below 0.01. State heatmaps use a log colour scale. Cells with no transition are white. A vector PDF is written only when the state count is at most 2000.

In [3]:
import re

# 读已经写好的 joint 计数和簇结果，不重新聚类。原有的 transition_blocks.png 不动。


def sig2(value):
    if not np.isfinite(value) or value < 0.01:
        return ""
    return f"{value:.2g}"


def cluster_spans(cluster_ids):
    spans = []
    start = 0
    for i in range(1, len(cluster_ids) + 1):
        if i == len(cluster_ids) or cluster_ids[i] != cluster_ids[start]:
            spans.append((int(cluster_ids[start]), start, i))
            start = i
    return spans


def mark_cluster_edges(ax, spans, half_pixel):
    # imshow 的格子中心在整数上，边界在半整数。pcolormesh 的边界在整数上。
    for _cluster, _start, end in spans[:-1]:
        edge = end - 0.5 if half_pixel else end
        ax.axhline(edge, color="k", lw=0.4)
        ax.axvline(edge, color="k", lw=0.4)
    if half_pixel:
        centers = [0.5 * (start + end - 1) for _c, start, end in spans]
    else:
        centers = [0.5 * (start + end) for _c, start, end in spans]
    labels = [str(c) for c, _s, _e in spans]
    ax.set_xticks(centers, labels, fontsize=6)
    ax.set_yticks(centers, labels, fontsize=6)


def log_limits(values):
    positive = values[np.isfinite(values) & (values > 0)]
    if positive.size == 0:
        return None
    vmin = float(positive.min())
    vmax = float(positive.max())
    if vmax <= vmin:
        vmax = vmin * 10.0
    return vmin, vmax


def save_png(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=DPI_BIG, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(" saved", path)


def save_vector_pdf(matrix, clusters, path, title):
    n = matrix.shape[0]
    if n > 2000:
        print(f"  skip pdf {path.name}: {n} states is above 2000")
        return
    show = np.ma.masked_where(~np.isfinite(matrix) | (matrix <= 0), matrix)
    limits = log_limits(show.filled(np.nan) if hasattr(show, "filled") else matrix)
    fig, ax = plt.subplots(figsize=(12, 12))
    if limits is None:
        ax.set_title(title + "\nno positive transitions")
    else:
        cmap = plt.get_cmap("viridis").copy()
        cmap.set_bad("white")
        grid = np.arange(n + 1)
        ax.pcolormesh(grid, grid, show, cmap=cmap, norm=LogNorm(*limits), shading="flat")
        ax.set_aspect("equal")
        ax.invert_yaxis()
        mark_cluster_edges(ax, cluster_spans(clusters), half_pixel=False)
        ax.set_xlabel("state at t+tau")
        ax.set_ylabel("state at t")
        ax.set_title(title)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, format="pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(" saved", path)


def draw_state_matrix(matrix, clusters, path, title, pdf_path):
    show = np.ma.masked_where(~np.isfinite(matrix) | (matrix <= 0), matrix)
    limits = log_limits(np.asarray(show.filled(np.nan)))
    fig, ax = plt.subplots(figsize=(12, 12))
    if limits is None:
        ax.set_title(title + "\nno positive transitions")
    else:
        cmap = plt.get_cmap("viridis").copy()
        cmap.set_bad("white")
        ax.imshow(show, cmap=cmap, norm=LogNorm(*limits), interpolation="nearest")
        mark_cluster_edges(ax, cluster_spans(clusters), half_pixel=True)
        ax.set_xlabel("state at t+tau")
        ax.set_ylabel("state at t")
        ax.set_title(title)
        fig.colorbar(ax.images[0], ax=ax, fraction=0.046, label="transition probability")
    save_png(fig, path)
    save_vector_pdf(matrix, clusters, pdf_path, title)


def draw_cluster_matrix(prob, counts, cluster_ids, tick_labels, path, title, blank_diagonal):
    show = prob.astype(np.float64, copy=True)
    show[show <= 0] = np.nan
    if blank_diagonal:
        np.fill_diagonal(show, np.nan)
    fig_side = max(8.0, min(36.0, 0.18 * len(tick_labels) + 2.0))
    fig, ax = plt.subplots(figsize=(fig_side, fig_side))
    finite = show[np.isfinite(show)]
    vmax = float(finite.max()) if finite.size else 1.0
    if vmax <= 0:
        vmax = 1.0
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")
    masked = np.ma.masked_invalid(show)
    image = ax.imshow(masked, cmap=cmap, vmin=0.0, vmax=vmax, interpolation="nearest")
    fig.colorbar(image, ax=ax, fraction=0.046, label="transition probability")
    n = len(tick_labels)
    fontsize = 7 if n <= 30 else 4
    for i in range(n):
        for j in range(n):
            text = sig2(show[i, j])
            if not text:
                continue
            color = "white" if show[i, j] > 0.55 * vmax else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=fontsize, color=color)
    ax.set_xticks(np.arange(n), tick_labels, fontsize=fontsize)
    ax.set_yticks(np.arange(n), tick_labels, fontsize=fontsize)
    ax.set_xlabel("cluster at t+tau")
    ax.set_ylabel("cluster at t")
    ax.set_title(title)
    save_png(fig, path)
    stem = path.with_suffix(".csv")
    ids = [str(int(c)) for c in cluster_ids]
    pd.DataFrame(prob, index=ids, columns=ids).to_csv(stem)
    pd.DataFrame(counts, index=ids, columns=ids).to_csv(stem.with_name(stem.stem + "_counts.csv"))


def draw_zoom(prob, counts, state_ids, cluster_id, n_states, path, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
    panels = (
        (prob, "with diagonal"),
        (np.where(np.eye(len(state_ids), dtype=bool), np.nan, prob), "diagonal blank"),
    )
    for ax, (matrix, note) in zip(axes, panels):
        work = np.array(matrix, dtype=np.float64, copy=True)
        work[~(np.isfinite(work) & (work > 0))] = np.nan
        show = np.ma.masked_invalid(work)
        finite = work[np.isfinite(work)]
        if finite.size == 0:
            ax.set_title(f"cluster {cluster_id}  {n_states} states\n{note}\nno positive transitions")
            continue
        limits = log_limits(work)
        cmap = plt.get_cmap("viridis").copy()
        cmap.set_bad("white")
        ax.imshow(show, cmap=cmap, norm=LogNorm(*limits), interpolation="nearest")
        ax.set_xticks(np.arange(len(state_ids)), state_ids, rotation=90, fontsize=6)
        ax.set_yticks(np.arange(len(state_ids)), state_ids, fontsize=6)
        ax.set_xlabel("state at t+tau")
        ax.set_ylabel("state at t")
        ax.set_title(f"cluster {cluster_id}  {n_states} states\n{note}")
    fig.suptitle(title_prefix)
    fig.tight_layout()
    save_png(fig, path)


def maybe_html(prob, counts, state_ids, clusters, path, title):
    if not MAKE_HTML:
        print(" MAKE_HTML is False; no html")
        return
    try:
        import plotly.graph_objects as go
    except ImportError:
        message = "plotly is not installed:\n  .venv\\Scripts\\python.exe -m pip install plotly"
        print(message)
        raise RuntimeError(message)
    n = len(state_ids)
    custom = np.dstack([
        np.broadcast_to(state_ids, (n, n)),
        np.broadcast_to(clusters, (n, n)),
        np.broadcast_to(state_ids, (n, n)).T,
        np.broadcast_to(clusters, (n, n)).T,
        counts,
    ])
    show = np.where(prob > 0, prob, np.nan)
    fig = go.Figure(data=go.Heatmap(
        z=show,
        x=[str(s) for s in state_ids],
        y=[str(s) for s in state_ids],
        customdata=custom,
        hovertemplate=("from state %{y} (cluster %{customdata[1]})<br>"
                       "to state %{x} (cluster %{customdata[3]})<br>"
                       "probability %{z:.4g}<br>count %{customdata[4]:.0f}<extra></extra>"),
        colorscale="Viridis",
        colorbar={"title": "probability"},
    ))
    fig.update_layout(title=title, width=900, height=900, xaxis_title="state at t+tau", yaxis_title="state at t")
    fig.update_yaxes(autorange="reversed")
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path)
    print(" saved", path)


pattern = re.compile(r"_tau(\d+)_infomap$")
jobs = []
for step7 in sorted(ROOT.glob("*/*/*_infomap/step7")):
    match = pattern.search(step7.parent.name)
    if match is None or not (step7 / "state_order.csv").exists():
        continue
    joint = step7.parent.parent / step7.parent.name.replace("_infomap", "_joint") / "step5" / "counts.npz"
    if not joint.exists():
        raise RuntimeError(f"missing counts: {joint}")
    jobs.append((step7, joint, int(match.group(1))))
print(f"transition figure jobs {len(jobs)}")

for step7, joint, tau in jobs:
    analysis, version = step7.parts[-4], step7.parts[-3]
    order = pd.read_csv(step7 / "state_order.csv")
    summary = pd.read_csv(step7 / "cluster_summary.csv")
    checks = pd.read_csv(step7 / "checks.csv").iloc[0]
    frames = pd.read_csv(step7 / "frame_clusters.csv", usecols=["state", "cluster"])
    counts = sparse.load_npz(joint).tocsr()
    states = order["state"].to_numpy(dtype=int)
    clusters = order["cluster"].to_numpy(dtype=int)
    visits = frames.loc[frames["state"] >= 0, "state"].value_counts()
    n_valid = int((frames["state"] >= 0).sum())
    prob_full, _row = row_normalize(counts)
    probability = np.asarray(prob_full[np.ix_(states, states)].toarray(), dtype=np.float64)
    count_sub = np.asarray(counts[np.ix_(states, states)].toarray(), dtype=np.float64)
    seconds = float(checks["tau_seconds"])
    backend = f"{checks['backend']} {checks['backend_version']}"
    head = (f"{analysis} {version}  tau={tau} frames ({seconds:.4f} s)  "
            f"{len(states)} states, {order['cluster'].nunique()} clusters  {backend}")
    out = step7 / "transition_blocks"
    tag = f"tau{tau}_selfloops{int(bool(KEEP_SELF_LOOPS))}"
    print(f"\n{head}")

    cluster_ids = pd.unique(clusters)
    index = {int(c): i for i, c in enumerate(cluster_ids)}
    lookup = np.full(counts.shape[0], -1, dtype=np.int32)
    lookup[states] = np.array([index[int(c)] for c in clusters], dtype=np.int32)
    coo = counts.tocoo()
    left = lookup[coo.row]
    right = lookup[coo.col]
    keep = (left >= 0) & (right >= 0)
    pooled = np.zeros((len(cluster_ids), len(cluster_ids)), dtype=np.float64)
    np.add.at(pooled, (left[keep], right[keep]), coo.data[keep])
    row_sum = pooled.sum(axis=1, keepdims=True)
    pooled_prob = np.divide(pooled, row_sum, out=np.zeros_like(pooled), where=row_sum > 0)
    info = summary.set_index("cluster")
    tick_labels = []
    for cluster in cluster_ids:
        n_states = int(info.loc[cluster, "n_states"]) if cluster in info.index else int(np.sum(clusters == cluster))
        n_frames = int(info.loc[cluster, "n_frames"]) if cluster in info.index else 0
        tick_labels.append(f"{int(cluster)}\n{n_states} st\n{n_frames} fr")
    draw_cluster_matrix(
        pooled_prob, pooled, cluster_ids, tick_labels, out / f"{tag}_clusters_with_diagonal.png",
        head + "\ncluster matrix, with diagonal", False,
    )
    draw_cluster_matrix(
        pooled_prob, pooled, cluster_ids, tick_labels, out / f"{tag}_clusters_diagonal_blank.png",
        head + "\ncluster matrix, diagonal blank; colour scale excludes the diagonal", True,
    )

    draw_state_matrix(
        probability, clusters, out / f"{tag}_states_dpi{DPI_BIG}_with_diagonal.png",
        head + "\nstates, with diagonal", out / f"{tag}_states_dpi{DPI_BIG}_with_diagonal.pdf",
    )
    no_diag = probability.copy()
    np.fill_diagonal(no_diag, np.nan)
    draw_state_matrix(
        no_diag, clusters, out / f"{tag}_states_dpi{DPI_BIG}_no_diagonal.png",
        head + "\nstates, diagonal removed", out / f"{tag}_states_dpi{DPI_BIG}_no_diagonal.pdf",
    )

    if FILTER_RARE:
        visit_of = visits.reindex(states).fillna(0.0).to_numpy()
        kept = visit_of >= MIN_VISITS
        dropped_states = states[~kept]
        dropped_frames = int(visits.reindex(dropped_states).fillna(0).sum())
        frac = dropped_frames / n_valid if n_valid else np.nan
        print(f"  rare filter MIN_VISITS={MIN_VISITS}: drop {int((~kept).sum())} states, "
              f"{dropped_frames}/{n_valid} valid frames ({frac:.4%})")
        if int(kept.sum()) == 0:
            print("  no states left after the rare filter")
        else:
            rare_prob = probability[np.ix_(kept, kept)]
            rare_clusters = clusters[kept]
            rare_head = head.replace(f"{len(states)} states", f"{int(kept.sum())} states")
            draw_state_matrix(
                rare_prob, rare_clusters,
                out / f"{tag}_states_dpi{DPI_BIG}_minvisits{MIN_VISITS}_with_diagonal.png",
                rare_head + f"\nstates with at least {MIN_VISITS} frames, with diagonal",
                out / f"{tag}_states_dpi{DPI_BIG}_minvisits{MIN_VISITS}_with_diagonal.pdf",
            )
            rare_blank = rare_prob.copy()
            np.fill_diagonal(rare_blank, np.nan)
            draw_state_matrix(
                rare_blank, rare_clusters,
                out / f"{tag}_states_dpi{DPI_BIG}_minvisits{MIN_VISITS}_no_diagonal.png",
                rare_head + f"\nstates with at least {MIN_VISITS} frames, diagonal removed",
                out / f"{tag}_states_dpi{DPI_BIG}_minvisits{MIN_VISITS}_no_diagonal.pdf",
            )
    else:
        print(" FILTER_RARE is False")

    ranked = summary.sort_values(["n_frames", "cluster"], ascending=[False, True])
    for cluster in ranked["cluster"].head(TOP_ZOOM):
        mask = clusters == int(cluster)
        if not np.any(mask):
            continue
        block = probability[np.ix_(mask, mask)]
        block_counts = count_sub[np.ix_(mask, mask)]
        members = states[mask]
        draw_zoom(
            block, block_counts, members, int(cluster), int(mask.sum()),
            out / f"{tag}_zoom_cluster{int(cluster)}.png",
            head + f"\nzoom cluster {int(cluster)}",
        )

    maybe_html(
        probability, count_sub, states, clusters,
        out / f"{tag}_states.html",
        head + "\nstates, with diagonal",
    )

payload_path = ROOT / "params.json"
if payload_path.exists():
    payload = json.loads(payload_path.read_text(encoding="utf-8"))
    payload["parameters"] = {k: (str(v) if isinstance(v, Path) else v) for k, v in PARAMS.items()}
    payload_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("updated", payload_path)


transition figure jobs 24



K pairs_FM1  tau=17 frames (0.1024 s)  680 states, 128 clusters  python-igraph 1.0.0


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_infomap\step7\transition_blocks\tau17_selfloops1_clusters_with_diagonal.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_infomap\step7\transition_blocks\tau17_selfloops1_clusters_diagonal_blank.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_infomap\step7\transition_blocks\tau17_selfloops1_states_dpi300_with_diagonal.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K\pairs_FM1\bins10_pcs1-2-3_vars-pcs_tau17_infomap\step7\transition_blocks\tau17_selfloops1_states_dpi300_with_diagonal.pdf


... (350 outputs left out)

 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_states_dpi300_minvisits50_no_diagonal.pdf
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster0.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster1.png
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster2.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster3.png
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster4.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\P\triple\bins10_pcs1-2-3_vars-pcs_tau7_infomap\step7\transition_blocks\tau7_selfloops1_zoom_cluster5.png
 MAKE_HTML is False; no html
updated Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\params.json


## Step 6 — K and P, numbers only

Loadings, autocorrelation, and cross-projection are written for each version. The day split is a reference from the same fish on different days. Nothing here is labelled consistent or inconsistent.


In [10]:
pc_index = [p - 1 for p in PCS_FOR_STATE]

def valid_rows(feat):
    return feat[np.isfinite(feat).all(axis=1)]


def project_videos(videos, mu, components):
    projected = []
    for video in videos:
        projected.append({
            "experiment": video["experiment"],
            "fps": video["fps"],
            "seg": video["seg"],
            "frames": video["frames"],
            "scores": score_frames(video["feat"], mu, components),
        })
    return projected


def outside_fraction(videos, edges):
    fracs = []
    for j in range(edges.shape[0]):
        values = np.concatenate([v["scores"][:, pc_index[j]] for v in videos])
        finite = np.isfinite(values)
        if finite.sum() == 0:
            fracs.append(np.nan)
            continue
        outside = finite & ((values < edges[j, 0]) | (values > edges[j, -1]))
        fracs.append(float(outside.sum() / finite.sum()))
    return fracs


def states_of(videos, edges, mode):
    per_video = []
    any_outside = []
    for video in videos:
        cols = [digitize(video["scores"][:, pc_index[j]], edges[j]) for j in range(len(pc_index))]
        finite = np.isfinite(video["scores"][:, pc_index]).all(axis=1)
        if mode == "joint":
            state = joint_from_bins(cols)
            any_outside.append(finite & (state < 0))
            per_video.append({"state": state, "seg": video["seg"], "fps": video["fps"]})
        else:
            per_video.append({"bins": cols, "seg": video["seg"], "fps": video["fps"]})
    return per_video, any_outside


def stay_of(per_video, mode, tau, n_states, pc_slot=None):
    counts = sparse.csr_matrix((n_states, n_states), dtype=np.float64)
    for video in per_video:
        series = video["state"] if mode == "joint" else video["bins"][pc_slot]
        part, _n = transition_matrix(series, video["seg"], tau, n_states)
        counts = counts + part
    return stay_fraction(counts)


def fit_half(videos):
    x = np.concatenate([valid_rows(v["feat"]) for v in videos], axis=0)
    mu, components, explained = fit_pca(x)
    scored = project_videos(videos, mu, components)
    stacked = np.concatenate([v["scores"][:, pc_index] for v in scored], axis=0)
    edges = np.vstack([bin_edges(stacked[:, j]) for j in range(stacked.shape[1])])
    return {"mu": mu, "components": components, "explained": explained, "videos": scored, "edges": edges}


def halves_of(videos):
    order = sorted(videos, key=lambda v: (M.parse_experiment(v["experiment"])[1], v["experiment"]))
    mid = max(len(order) // 2, 1)
    if mid >= len(order):
        mid = len(order) - 1
    return order[:mid], order[mid:]


summary_rows = []
VERSIONS = list(PAIRS) + [("triple", None, FISH)]

for version, _pair, _fish in VERSIONS:
    k_item = STORE[("K", version)]
    p_item = STORE[("P", version)]
    if k_item["feature_names"] != p_item["feature_names"]:
        raise RuntimeError(f"{version}: K and P feature order differ")
    print(f"{version} shared feature order ({len(k_item['feature_names'])})")
    n = min(N_CMP, k_item["components"].shape[1], p_item["components"].shape[1])
    sim = cosine_abs(k_item["components"], p_item["components"], n)
    match = best_matches(sim)
    match.insert(0, "version", version)
    out = loc6(version, "a_loadings")
    pd.DataFrame(sim, index=[f"K_PC{i+1}" for i in range(n)],
                 columns=[f"P_PC{i+1}" for i in range(n)]).to_csv(out / "cosine_abs.csv")
    match.to_csv(out / "best_match.csv", index=False)
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(sim, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(n), [f"P{i+1}" for i in range(n)])
    ax.set_yticks(range(n), [f"K{i+1}" for i in range(n)])
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if sim[i, j] > 0.6 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, label="absolute cosine")
    ax.set_title(f"{version} loading |cosine|\nsame feature order; absolute cosine")
    save_fig(fig, out / "cosine_abs.png")
    m = min(len(k_item["explained"]), len(p_item["explained"]))
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(np.arange(1, m + 1), k_item["explained"][:m], marker="o", ms=3, label="K")
    ax.plot(np.arange(1, m + 1), p_item["explained"][:m], marker="o", ms=3, label="P")
    ax.set_xlabel("component")
    ax.set_ylabel("variance fraction")
    ax.legend()
    ax.set_title(f"{version} variance fraction")
    save_fig(fig, out / "variance_fraction.png")
    print(f"\n{version} best |cosine| per K component")
    print(match.to_string(index=False))

    out_b = loc6(version, "b_acf")
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)
    time_rows = []
    for ax, row in zip(axes.ravel(), match.itertuples(index=False)):
        ik, ip = int(row.pc_row) - 1, int(row.pc_col) - 1
        ax.plot(k_item["acf_lag_s"], k_item["acf"][ik], label=f"K PC{ik+1}", color="#c0392b")
        ax.plot(p_item["acf_lag_s"], p_item["acf"][ip], label=f"P PC{ip+1}", color="#1a5276")
        ax.axhline(np.exp(-1), color="0.5", lw=0.6, ls="--")
        ax.set_title(f"K{ik+1}–P{ip+1}  |cos| {row.abs_cosine:.2f}")
        ax.legend(fontsize=7)
        time_rows.append({
            "version": version, "k_pc": ik + 1, "p_pc": ip + 1,
            "abs_cosine": row.abs_cosine,
            "k_time_to_1_over_e_s": k_item["tau_e"][ik],
            "p_time_to_1_over_e_s": p_item["tau_e"][ip],
        })
    for ax in axes[-1]:
        ax.set_xlabel("lag (s)")
    fig.suptitle(f"{version} ACF of each K component and its highest-|cosine| P component")
    fig.tight_layout()
    save_fig(fig, out_b / "acf_matched.png")
    pd.DataFrame(time_rows).to_csv(out_b / "correlation_time.csv", index=False)

    transfer_rows = []
    for source_name, target_name, source, target in (
        ("K", "P", k_item, p_item),
        ("P", "K", p_item, k_item),
    ):
        projected = project_videos(target["videos"], source["mu"], source["components"])
        fracs = outside_fraction(projected, source["edges"])
        own = project_videos(source["videos"], source["mu"], source["components"])
        own_fracs = outside_fraction(own, source["edges"])
        for mode in (["joint"] if DO_JOINT else []) + (["per_pc"] if DO_PER_PC else []):
            src_videos, _o = states_of(own, source["edges"], mode)
            dst_videos, _dst_out = states_of(projected, source["edges"], mode)
            if mode == "joint":
                n_states = N_BINS ** len(pc_index)
                p_src = occupancy(np.concatenate([v["state"] for v in src_videos]), n_states)
                p_dst = occupancy(np.concatenate([v["state"] for v in dst_videos]), n_states)
                distance = js_distance(p_src, p_dst)
                finite = np.concatenate([np.isfinite(v["scores"][:, pc_index]).all(axis=1) for v in projected])
                state = np.concatenate([v["state"] for v in dst_videos])
                outside_any = float(((state < 0) & finite).sum() / max(int(finite.sum()), 1))
                fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
                for ax, counts, label in (
                    (axes[0], p_src, f"{source_name} on its own grid"),
                    (axes[1], p_dst, f"{target_name} projected onto {source_name}"),
                ):
                    order = np.argsort(counts)[::-1]
                    keep = order[counts[order] > 0][:40]
                    ax.bar(np.arange(len(keep)), counts[keep], color="0.25")
                    ax.set_title(label)
                    ax.set_xlabel("occupied states, top 40")
                    ax.set_ylabel("frames")
                fig.suptitle(f"{version} {source_name} grid  JS distance {distance:.4f}\n"
                             "Outside frames are excluded from both histograms")
                out_c = loc6(version, "c_transfer")
                save_fig(fig, out_c / f"occupancy_{source_name}_to_{target_name}.png")
                for tau in TAUS:
                    transfer_rows.append({
                        "version": version, "direction": f"{source_name}_to_{target_name}",
                        "mode": mode, "pc": "joint",
                        "js_distance": distance,
                        "tau_frames": tau,
                        "stay_source": stay_of(src_videos, "joint", tau, n_states),
                        "stay_projected": stay_of(dst_videos, "joint", tau, n_states),
                        "outside_fraction_any_pc": outside_any,
                    })
            else:
                out_c = loc6(version, "c_transfer")
                for j, pc in enumerate(PCS_FOR_STATE):
                    p_src = occupancy(np.concatenate([v["bins"][j] for v in src_videos]), N_BINS)
                    p_dst = occupancy(np.concatenate([v["bins"][j] for v in dst_videos]), N_BINS)
                    distance = js_distance(p_src, p_dst)
                    for tau in TAUS:
                        transfer_rows.append({
                            "version": version, "direction": f"{source_name}_to_{target_name}",
                            "mode": mode, "pc": pc,
                            "js_distance": distance,
                            "tau_frames": tau,
                            "stay_source": stay_of(src_videos, "per_pc", tau, N_BINS, j),
                            "stay_projected": stay_of(dst_videos, "per_pc", tau, N_BINS, j),
                            "outside_fraction_this_pc": fracs[j],
                            "source_outside_fraction_this_pc": own_fracs[j],
                        })
        print(f"{version} {source_name}->{target_name} outside fraction per PC {np.round(fracs, 4)}  "
              f"source outside {np.round(own_fracs, 4)}")
    transfer = pd.DataFrame(transfer_rows)
    loc6(version, "c_transfer").mkdir(parents=True, exist_ok=True)
    transfer.to_csv(loc6(version, "c_transfer") / "transfer.csv", index=False)

    ref_rows = []
    if DO_DAY_REFERENCE:
        for group_name in ("K", "P"):
            videos = STORE[(group_name, version)]["videos"]
            first, second = halves_of(videos)
            # Refit needs the original features. project_videos replaced nothing on STORE videos.
            a = fit_half(first)
            b = fit_half(second)
            n_ref = min(N_CMP, a["components"].shape[1], b["components"].shape[1])
            sim_ref = cosine_abs(a["components"], b["components"], n_ref)
            ref_match = best_matches(sim_ref)
            ref_out = loc6(version, "d_day_reference")
            pd.DataFrame(sim_ref, index=[f"dayA_PC{i+1}" for i in range(n_ref)],
                         columns=[f"dayB_PC{i+1}" for i in range(n_ref)]).to_csv(
                ref_out / f"{group_name}_cosine_abs.csv")
            names_a = [v["experiment"] for v in first]
            names_b = [v["experiment"] for v in second]
            print(f"{version} {group_name} day split {names_a} vs {names_b}  "
                  f"mean best |cosine| {ref_match['abs_cosine'].mean():.3f}")
            for direction, src in (("dayA_to_dayB", a), ("dayB_to_dayA", b)):
                source_videos = first if src is a else second
                target_videos = second if src is a else first
                projected = project_videos(target_videos, src["mu"], src["components"])
                own = project_videos(source_videos, src["mu"], src["components"])
                src_states, _ = states_of(own, src["edges"], "joint")
                dst_states, _ = states_of(projected, src["edges"], "joint")
                n_states = N_BINS ** len(pc_index)
                distance = js_distance(
                    occupancy(np.concatenate([v["state"] for v in src_states]), n_states),
                    occupancy(np.concatenate([v["state"] for v in dst_states]), n_states),
                )
                ref_rows.append({
                    "version": version, "group": group_name, "direction": direction,
                    "day_a": " ".join(names_a), "day_b": " ".join(names_b),
                    "mean_best_abs_cosine": float(ref_match["abs_cosine"].mean()),
                    "min_best_abs_cosine": float(ref_match["abs_cosine"].min()),
                    "js_distance_joint": distance,
                    "note": "same fish on different days; not an independent sample; reference only",
                })
        pd.DataFrame(ref_rows).to_csv(loc6(version, "d_day_reference") / "day_reference.csv", index=False)

    kp = transfer[(transfer["mode"] == "joint") & (transfer["tau_frames"] == TAUS[0])]
    row = {
        "version": version,
        "mean_best_abs_cosine_K_P": float(match["abs_cosine"].mean()),
        "min_best_abs_cosine_K_P": float(match["abs_cosine"].min()),
    }
    for direction in ("K_to_P", "P_to_K"):
        part = kp[kp["direction"] == direction]
        row[f"js_joint_{direction}"] = float(part["js_distance"].iloc[0]) if len(part) else np.nan
        row[f"stay_source_{direction}"] = float(part["stay_source"].iloc[0]) if len(part) else np.nan
        row[f"stay_projected_{direction}"] = float(part["stay_projected"].iloc[0]) if len(part) else np.nan
    if ref_rows:
        ref = pd.DataFrame(ref_rows)
        for group_name in ("K", "P"):
            part = ref[ref["group"] == group_name]
            row[f"mean_best_abs_cosine_{group_name}_days"] = float(part["mean_best_abs_cosine"].mean())
            row[f"js_joint_{group_name}_days_mean"] = float(part["js_distance_joint"].mean())
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(loc6("_all", "e_summary") / "summary.csv", index=False)
# Also one summary next to each version.
for version, _pair, _fish in VERSIONS:
    part = summary[summary["version"] == version]
    part.to_csv(loc6(version, "e_summary") / "summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(summary))
axes[0].bar(x - 0.2, summary["mean_best_abs_cosine_K_P"], width=0.2, label="K vs P")
if "mean_best_abs_cosine_K_days" in summary:
    axes[0].bar(x, summary["mean_best_abs_cosine_K_days"], width=0.2, label="K days")
    axes[0].bar(x + 0.2, summary["mean_best_abs_cosine_P_days"], width=0.2, label="P days")
axes[0].set_xticks(x, summary["version"], rotation=20)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("mean best |cosine|")
axes[0].legend(fontsize=8)
axes[0].set_title("Loading similarity")
axes[1].bar(x - 0.2, summary["js_joint_K_to_P"], width=0.2, label="K to P")
if "js_joint_K_days_mean" in summary:
    axes[1].bar(x, summary["js_joint_K_days_mean"], width=0.2, label="K days")
    axes[1].bar(x + 0.2, summary["js_joint_P_days_mean"], width=0.2, label="P days")
axes[1].set_xticks(x, summary["version"], rotation=20)
axes[1].set_ylabel("JS distance")
axes[1].legend(fontsize=8)
axes[1].set_title("Joint-state occupancy")
fig.suptitle("Numbers only. Day bars are the same fish on two days, not independent samples.")
fig.tight_layout()
save_fig(fig, loc6("_all", "e_summary") / "summary.png")
print("\nsummary")
print(summary.to_string(index=False))

frame_counts = {}
for (analysis, version), item in STORE.items():
    frame_counts.setdefault(analysis, {})[version] = int(sum(
        np.isfinite(v["scores"][:, 0]).sum() for v in item["videos"]))
payload = {
    "date": time.strftime("%Y-%m-%d %H:%M"),
    "run_name": RUN_NAME,
    "output_root": str(ROOT),
    "parameters": {k: (str(v) if isinstance(v, Path) else v) for k, v in PARAMS.items()},
    "valid_frames": frame_counts,
    "infomap": INFOMAP_INFO,
}
(ROOT / "params.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote", ROOT / "params.json")


pairs_FM1 shared feature order (14)
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\pairs_FM1\a_loadings\cosine_abs.png
 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\pairs_FM1\a_loadings\variance_fraction.png

pairs_FM1 best |cosine| per K component
  version  pc_row  pc_col  abs_cosine
pairs_FM1       1       2    0.766515
pairs_FM1       2       1    0.766437
pairs_FM1       3       3    0.736718
pairs_FM1       4       4    0.819821
pairs_FM1       5       5    0.785911
pairs_FM1       6       6    0.824796


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\pairs_FM1\b_acf\acf_matched.png


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\pairs_FM1\c_transfer\occupancy_K_to_P.png


pairs_FM1 K->P outside fraction per PC [0.     0.     0.0003]  source outside [0. 0. 0.]


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\pairs_FM1\c_transfer\occupancy_P_to_K.png


pairs_FM1 P->K outside fraction per PC [0.001  0.0227 0.    ]  source outside [0. 0. 0.]


... (23 outputs left out)

triple P->K outside fraction per PC [0.0001 0.0002 0.0003]  source outside [0. 0. 0.]


triple K day split ['20251225_K_1_1_FCMC'] vs ['20251225_K_1_2_FCMC']  mean best |cosine| 0.735


triple P day split ['20260112_P_1_1_FCMC'] vs ['20260113_P_1_2_FCMC']  mean best |cosine| 0.694


 saved Z:\medaka\runs\behavior_features\pose_pca\20261106_v2\K_vs_P\_all\e_summary\summary.png

summary
   version  mean_best_abs_cosine_K_P  min_best_abs_cosine_K_P  js_joint_K_to_P  stay_source_K_to_P  stay_projected_K_to_P  js_joint_P_to_K  stay_source_P_to_K  stay_projected_P_to_K  mean_best_abs_cosine_K_days  js_joint_K_days_mean  mean_best_abs_cosine_P_days  js_joint_P_days_mean
 pairs_FM1                  0.783366                 0.736718         0.440238            0.721906               0.784704         0.423329            0.820999               0.753992                     0.977494              0.221321                     0.941598              0.290804
 pairs_FM2                  0.854445                 0.761356         0.405238            0.760604               0.814051         0.415120            0.855777               0.810833                     0.928074              0.204397                     0.851389              0.318708
pairs_M1M2                  0.907980        

## Step 7 — manual labels

`DO_VALIDATION` is off, so this cell does not read the BORIS files. Clustering, tau, bins, and the choice of components do not read labels. This is the only cell that can.

Fill `LABEL_PAIR_MAP` before turning the switch on. FM1 is compared with M1's labels, FM2 with M2's labels, and M1M2 with labels between the two males. The map is learned on `DEV_SETS` and applied unchanged to `TEST_SETS`, then the two sides are swapped. The split is by group, not by frame or by video.

Cluster numbers are taken from the KP clustering. A cluster number in the K run is not the same cluster as that number in the P run.

Development numbers are a reference. Test numbers are the ones to read. The cell writes the numbers and the figures and does not decide whether the clusters match the labels.

A frame with more than one mapped behaviour is kept as one label, with the names joined by `+`. An event hit is an overlap of at least `OVERLAP_THR` of the reference event after both intervals are expanded by `TOL_FRAMES`.

In [4]:
import re

# 这一格是 notebook 里唯一读取手动标记的地方。默认不运行。
# 簇编号来自 KP 的聚类：K 和 P 分开聚类时，同一个编号不是同一个簇。


def video_group(experiment):
    match = re.search(r"_([A-Z]+)_", experiment)
    if match is None:
        raise RuntimeError(f"cannot read the group from {experiment}")
    return match.group(1)


def boris_file(experiment):
    hits = sorted(BORIS_DIR.glob(f"**/{experiment}.csv"))
    if len(hits) != 1:
        raise FileNotFoundError(f"{experiment}: found {len(hits)} BORIS files under {BORIS_DIR}")
    return hits[0]


def expand_labels(experiment, subjects, behavior_map):
    # 一个帧只留一个标签。同一帧有多个映射标签时，用 "+" 连起来，并单独计数。
    table = pd.read_csv(boris_file(experiment))
    subjects = {str(s).casefold() for s in subjects}
    mapped = {str(k).casefold(): v for k, v in behavior_map.items()}
    table = table[table["Subject"].astype(str).str.casefold().isin(subjects)].copy()
    offset = table["Image index"] - table["Time"] * 166.0
    print(f"  {experiment} image index minus time*166: median {np.nanmedian(offset):.3f}")
    active = {}
    unclosed = 0
    orphan_stop = 0
    for subject, part in table.groupby("Subject", sort=False):
        for behavior, rows in part.groupby("Behavior", sort=False):
            label = mapped.get(str(behavior).casefold())
            if label is None:
                continue
            stack = []
            ordered = rows.sort_values("Image index")
            kinds = ordered["Behavior type"].to_numpy()
            indexes = np.rint(ordered["Image index"].to_numpy(dtype=float)).astype(int)
            for kind, frame in zip(kinds, indexes):
                kind = str(kind)
                frame = int(frame)
                if kind.upper() == "START":
                    stack.append(frame)
                elif kind.upper() == "STOP":
                    if not stack:
                        orphan_stop += 1
                        continue
                    start = stack.pop()
                    for value in range(min(start, frame), max(start, frame) + 1):
                        active.setdefault(value, set()).add(label)
                elif kind.upper() == "POINT":
                    active.setdefault(frame, set()).add(label)
            unclosed += len(stack)
    print(f"  {experiment} unclosed START {unclosed}  STOP without START {orphan_stop}")
    multi = sum(len(v) > 1 for v in active.values())
    rows = [{"experiment": experiment, "frame": frame, "label": "+".join(sorted(labels))}
            for frame, labels in sorted(active.items())]
    print(f"  {experiment} labelled frames {len(rows)}  frames with more than one mapped label {multi}")
    return pd.DataFrame(rows), multi


def label_runs(frames, labels):
    order = np.argsort(frames, kind="mergesort")
    frames = np.asarray(frames)[order]
    labels = np.asarray(labels)[order]
    runs = []
    if len(frames) == 0:
        return runs
    start = 0
    for i in range(1, len(frames)):
        if labels[i] != labels[i - 1] or frames[i] != frames[i - 1] + 1:
            runs.append((labels[start], int(frames[start]), int(frames[i - 1])))
            start = i
    runs.append((labels[start], int(frames[start]), int(frames[-1])))
    return runs


def overlap_frames(a0, a1, b0, b1, tol):
    left = max(a0 - tol, b0 - tol)
    right = min(a1 + tol, b1 + tol)
    return max(0, right - left + 1)


def event_hits(reference, candidates, tol, threshold):
    used = set()
    hits = 0
    for _label, a0, a1 in reference:
        best_i = None
        best = 0
        for i, (_lab, b0, b1) in enumerate(candidates):
            if i in used:
                continue
            length = overlap_frames(a0, a1, b0, b1, tol)
            if length > best:
                best = length
                best_i = i
        if best_i is not None and best / max(a1 - a0 + 1, 1) >= threshold:
            used.add(best_i)
            hits += 1
    return hits


def fit_mapping(dev):
    rows = []
    mapping = {}
    for cluster, part in dev.groupby("cluster"):
        counts = part["label"].value_counts()
        top = str(counts.index[0])
        purity = float(counts.iloc[0] / counts.sum())
        assigned = top if purity >= MIN_PURITY else "unassigned"
        mapping[int(cluster)] = assigned
        rows.append({
            "cluster": int(cluster),
            "n_labelled_frames": int(counts.sum()),
            "top_label": top,
            "top_count": int(counts.iloc[0]),
            "purity": purity,
            "assigned_label": assigned,
        })
    return mapping, pd.DataFrame(rows)


def purity_tables(data, mapping):
    summary = []
    long_rows = []
    for cluster, part in data.groupby("cluster"):
        counts = part["label"].value_counts()
        top = str(counts.index[0])
        summary.append({
            "cluster": int(cluster),
            "n_frames": int(len(part)),
            "top_label": top,
            "purity": float(counts.iloc[0] / counts.sum()),
            "frozen_label": mapping.get(int(cluster), "unassigned"),
        })
        for label, count in counts.items():
            long_rows.append({
                "cluster": int(cluster),
                "label": label,
                "n_frames": int(count),
                "proportion": float(count / len(part)),
            })
    return pd.DataFrame(summary), pd.DataFrame(long_rows)


def ordered_clusters(data, mapping):
    present = sorted(int(c) for c in data["cluster"].unique())
    return sorted(present, key=lambda c: (mapping.get(c, "unassigned"), c))


def confusion_frame(data, columns):
    labels = sorted(data["label"].unique())
    matrix = np.zeros((len(labels), len(columns)), dtype=np.float64)
    row = {label: i for i, label in enumerate(labels)}
    col = {key: i for i, key in enumerate(columns)}
    for label, key in zip(data["label"], data["column"]):
        matrix[row[label], col[key]] += 1
    return labels, matrix


def draw_confusion(matrix, rows, columns, path, title, normalize):
    show = matrix.copy()
    if normalize:
        denom = show.sum(axis=1, keepdims=True)
        show = np.divide(show, denom, out=np.zeros_like(show), where=denom > 0)
    side_w = max(6.0, min(36.0, 0.28 * len(columns) + 2.0))
    side_h = max(4.0, min(24.0, 0.28 * len(rows) + 2.0))
    fig, ax = plt.subplots(figsize=(side_w, side_h))
    image = ax.imshow(show, cmap="viridis", vmin=0, interpolation="nearest",
                      aspect="auto", vmax=1 if normalize else None)
    fig.colorbar(image, ax=ax, fraction=0.046, label="row fraction" if normalize else "frames")
    ax.set_xticks(np.arange(len(columns)), [str(c) for c in columns], rotation=90, fontsize=6)
    ax.set_yticks(np.arange(len(rows)), rows, fontsize=7)
    ax.set_xlabel("column")
    ax.set_ylabel("manual label")
    ax.set_title(title)
    save_fig(fig, path)


def score_table(manual, predicted):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    labels = sorted(set(manual) | set(predicted))
    precision, recall, f1, support = precision_recall_fscore_support(
        manual, predicted, labels=labels, zero_division=0,
    )
    table = pd.DataFrame({
        "label": labels,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })
    majority = pd.Series(manual).value_counts().index[0]
    baseline = float(np.mean(np.asarray(manual) == majority))
    accuracy = float(accuracy_score(manual, predicted))
    return table, accuracy, baseline, str(majority)


def agreement(manual, clusters, courtship, attack):
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    rows = [{
        "subset": "all labelled frames",
        "n_frames": int(len(manual)),
        "ARI": float(adjusted_rand_score(manual, clusters)),
        "NMI": float(normalized_mutual_info_score(manual, clusters)),
    }]
    wanted = {str(x) for x in list(courtship) + list(attack)}
    if not wanted:
        print("  COURTSHIP_LABELS and ATTACK_LABELS are empty; the courtship/attack ARI was not computed")
        return pd.DataFrame(rows)
    keep = [any(part in wanted for part in str(label).split("+")) for label in manual]
    keep = np.asarray(keep)
    if int(keep.sum()) == 0:
        print("  no courtship or attack frames under the configured label names")
        rows.append({"subset": "courtship or attack", "n_frames": 0, "ARI": np.nan, "NMI": np.nan})
        return pd.DataFrame(rows)
    rows.append({
        "subset": "courtship or attack",
        "n_frames": int(keep.sum()),
        "ARI": float(adjusted_rand_score(np.asarray(manual)[keep], np.asarray(clusters)[keep])),
        "NMI": float(normalized_mutual_info_score(np.asarray(manual)[keep], np.asarray(clusters)[keep])),
    })
    return pd.DataFrame(rows)


def event_table(data):
    rows = []
    labels = sorted(set(data["label"]) | set(data["predicted"]))
    for label in labels:
        manual_runs = []
        predicted_runs = []
        for _experiment, part in data.groupby("experiment"):
            manual = part[part["label"] == label]
            predicted = part[part["predicted"] == label]
            manual_runs += [(label, a, b) for lab, a, b in label_runs(manual["frame"], manual["label"])]
            predicted_runs += [(label, a, b) for lab, a, b in label_runs(predicted["frame"], predicted["predicted"])]
        recall_hit = event_hits(manual_runs, predicted_runs, TOL_FRAMES, OVERLAP_THR)
        precision_hit = event_hits(predicted_runs, manual_runs, TOL_FRAMES, OVERLAP_THR)
        rows.append({
            "label": label,
            "manual_events": len(manual_runs),
            "predicted_events": len(predicted_runs),
            "event_recall": recall_hit / len(manual_runs) if manual_runs else np.nan,
            "event_precision": precision_hit / len(predicted_runs) if predicted_runs else np.nan,
        })
    return pd.DataFrame(rows)


def draw_timeline(part, path, title, fps):
    labels = sorted(set(part["label"].dropna()) | set(part["predicted"].dropna()))
    cmap = plt.get_cmap("tab20")
    color = {label: cmap(i % 20) for i, label in enumerate(labels)}
    color["unassigned"] = (0.55, 0.55, 0.55, 1.0)
    fig, axes = plt.subplots(2, 1, figsize=(14, 3.4), sharex=True)
    for ax, column, name in ((axes[0], "label", "manual"), (axes[1], "predicted", "predicted")):
        runs = label_runs(part["frame"], part[column].to_numpy())
        for label, start, end in runs:
            ax.broken_barh([(start / fps, (end - start + 1) / fps)], (0, 1), facecolors=color.get(label, "0.5"))
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_ylabel(name)
    axes[1].set_xlabel("time (s)")
    handles = [plt.Line2D([0], [0], color=color[label], lw=4, label=str(label)) for label in labels]
    if handles:
        axes[0].legend(handles=handles, fontsize=6, ncol=6, loc="upper right")
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, path)


def bootstrap_scores(videos, labels):
    # 按视频重抽。这些视频不是独立样本。
    from sklearn.metrics import precision_recall_fscore_support
    rng = np.random.default_rng(INFOMAP_SEED)
    store = {label: {"precision": [], "recall": []} for label in labels}
    n = len(videos)
    if n == 0:
        return pd.DataFrame()
    for _draw in range(BOOT):
        pick = rng.choice(n, size=n, replace=True)
        part = pd.concat([videos[i] for i in pick], ignore_index=True)
        precision, recall, _f1, _support = precision_recall_fscore_support(
            part["label"], part["predicted"], labels=labels, zero_division=0,
        )
        for label, prec, rec in zip(labels, precision, recall):
            store[label]["precision"].append(prec)
            store[label]["recall"].append(rec)
    rows = []
    for label, values in store.items():
        for metric, series in values.items():
            rows.append({
                "label": label,
                "metric": metric,
                "p2.5": float(np.percentile(series, 2.5)),
                "p50": float(np.percentile(series, 50)),
                "p97.5": float(np.percentile(series, 97.5)),
                "n_videos": n,
                "note": "videos are not independent samples",
            })
    return pd.DataFrame(rows)


def shift_ari(videos):
    from sklearn.metrics import adjusted_rand_score
    rng = np.random.default_rng(INFOMAP_SEED + 1)
    scores = []
    for _repeat in range(SHIFT_N):
        manuals = []
        clusters = []
        for part in videos:
            frames = part["frame"].to_numpy()
            if len(frames) == 0:
                continue
            span = int(frames.max() - frames.min())
            delta = int(rng.integers(-max(span, 1), max(span, 1)))
            if delta == 0:
                delta = 1
            lookup = dict(zip(frames.tolist(), part["cluster"].tolist()))
            shifted = [lookup.get(int(frame) - delta) for frame in frames]
            keep = [value is not None for value in shifted]
            if not any(keep):
                continue
            manuals.extend(part["label"].to_numpy()[keep].tolist())
            clusters.extend([shifted[i] for i, flag in enumerate(keep) if flag])
        if len(set(manuals)) < 2 or len(set(clusters)) < 2:
            scores.append(np.nan)
        else:
            scores.append(float(adjusted_rand_score(manuals, clusters)))
    series = np.asarray(scores, dtype=np.float64)
    finite = series[np.isfinite(series)]
    print(f"  shifted ARI x{SHIFT_N}: n {len(finite)}  "
          f"mean {np.mean(finite) if len(finite) else np.nan:.4f}  "
          f"range {np.min(finite) if len(finite) else np.nan:.4f} .. "
          f"{np.max(finite) if len(finite) else np.nan:.4f}")
    return pd.DataFrame({"ARI": series})


def write_split(data, mapping, out, role, fps, examples):
    out.mkdir(parents=True, exist_ok=True)
    summary, long_rows = purity_tables(data, mapping)
    summary.to_csv(out / "purity_summary.csv", index=False)
    long_rows.to_csv(out / "purity_by_label.csv", index=False)
    work = data.copy()
    work["column"] = work["cluster"]
    row_labels, matrix = confusion_frame(work, ordered_clusters(work, mapping))
    columns = ordered_clusters(work, mapping)
    prefix = "development reference" if role == "dev" else "test"
    draw_confusion(matrix, row_labels, columns, out / "confusion_counts.png",
                   f"{prefix}\nmanual label vs cluster, counts", False)
    draw_confusion(matrix, row_labels, columns, out / "confusion_row_fraction.png",
                   f"{prefix}\nmanual label vs cluster, row-normalised", True)
    pd.DataFrame(matrix, index=row_labels, columns=columns).to_csv(out / "confusion_counts.csv")
    work["predicted"] = work["cluster"].map(lambda c: mapping.get(int(c), "unassigned"))
    pred_labels = sorted(work["predicted"].unique())
    work["column"] = work["predicted"]
    manual_labels, mapped = confusion_frame(work, pred_labels)
    draw_confusion(mapped, manual_labels, pred_labels, out / "confusion_mapped_counts.png",
                   f"{prefix}\nmanual label vs mapped label, counts", False)
    draw_confusion(mapped, manual_labels, pred_labels, out / "confusion_mapped_row_fraction.png",
                   f"{prefix}\nmanual label vs mapped label, row-normalised", True)
    scores, accuracy, baseline, majority = score_table(work["label"], work["predicted"])
    scores.to_csv(out / "class_scores.csv", index=False)
    pd.DataFrame([{
        "accuracy": accuracy,
        "majority_class": majority,
        "majority_baseline_accuracy": baseline,
    }]).to_csv(out / "accuracy.csv", index=False)
    print(f"  {role} accuracy {accuracy:.4f}  majority baseline ({majority}) {baseline:.4f}")
    agree = agreement(work["label"].tolist(), work["cluster"].astype(str).tolist(),
                      COURTSHIP_LABELS, ATTACK_LABELS)
    agree.to_csv(out / "ari_nmi.csv", index=False)
    print(agree.to_string(index=False))
    events = event_table(work)
    events.to_csv(out / "events.csv", index=False)
    print(events.to_string(index=False))
    if role == "test":
        videos = [part for _name, part in work.groupby("experiment")]
        bootstrap_scores(videos, sorted(work["label"].unique())).to_csv(out / "bootstrap.csv", index=False)
        shift_ari(videos).to_csv(out / "shift_ari.csv", index=False)
        shown = 0
        for experiment, part in work.groupby("experiment"):
            if shown >= N_EXAMPLE:
                break
            draw_timeline(part, out / f"timeline_{experiment}.png",
                          f"{experiment}\nmanual label and mapped label", fps)
            shown += 1


def run_validation():
    if not LABEL_PAIR_MAP:
        raise RuntimeError(
            "LABEL_PAIR_MAP is empty. Fill the subject and behaviour map, then set DO_VALIDATION = True."
        )
    directions = [(list(DEV_SETS), list(TEST_SETS)), (list(TEST_SETS), list(DEV_SETS))]
    pattern = re.compile(r"_tau(\d+)_infomap$")
    for step7 in sorted(ROOT.glob("KP/*/*_infomap/step7")):
        match = pattern.search(step7.parent.name)
        if match is None or not (step7 / "frame_clusters.csv").exists():
            continue
        version = step7.parts[-3]
        if version not in LABEL_PAIR_MAP:
            print(f"skip {version}: it is not in LABEL_PAIR_MAP")
            continue
        spec = LABEL_PAIR_MAP[version]
        tau = int(match.group(1))
        fps = float(pd.read_csv(step7 / "checks.csv").iloc[0]["tau_frames"]) / float(
            pd.read_csv(step7 / "checks.csv").iloc[0]["tau_seconds"]
        )
        frames = pd.read_csv(step7 / "frame_clusters.csv")
        print(f"\nvalidation {version} tau={tau} from the KP clustering")
        labelled_parts = []
        multi_total = 0
        for experiment in sorted(frames["experiment"].unique()):
            part, multi = expand_labels(experiment, spec["subjects"], spec["behaviors"])
            labelled_parts.append(part)
            multi_total += multi
        labels = pd.concat(labelled_parts, ignore_index=True)
        print(f"  frames with more than one mapped label, all videos: {multi_total}")
        print(labels["label"].value_counts().to_string())
        merged = labels.merge(frames, on=["experiment", "frame"], how="left")
        matched = merged["cluster"].ge(0)
        missing = ~matched
        print(f"  labelled frames {len(merged)}  matched to a cluster {int(matched.sum())}  "
              f"unmatched {int(missing.sum())} ({missing.mean():.4%})")
        usable = merged.loc[matched].copy()
        usable["cluster"] = usable["cluster"].astype(int)
        usable["group"] = usable["experiment"].map(video_group)
        for dev_groups, test_groups in directions:
            name = f"{'+'.join(dev_groups)}_to_{'+'.join(test_groups)}"
            dev = usable[usable["group"].isin(dev_groups)]
            test = usable[usable["group"].isin(test_groups)]
            print(f"\n{name}  dev frames {len(dev)}  test frames {len(test)}")
            print("  dev videos", sorted(dev["experiment"].unique()))
            print("  test videos", sorted(test["experiment"].unique()))
            if dev.empty or test.empty:
                raise RuntimeError(f"{name} has an empty side")
            mapping, mapping_table = fit_mapping(dev)
            out = ROOT / "validation" / name / version / f"tau{tau}"
            out.mkdir(parents=True, exist_ok=True)
            mapping_table.to_csv(out / "cluster_label_map.csv", index=False)
            write_split(dev, mapping, out / "dev", "dev", fps, N_EXAMPLE)
            write_split(test, mapping, out / "test", "test", fps, N_EXAMPLE)


if not DO_VALIDATION:
    print("DO_VALIDATION is False. Manual labels are not read.")
else:
    run_validation()


DO_VALIDATION is False. Manual labels are not read.
